# agent

> One routed conversation with tools, approvals, activity, and usage.

`Agent` binds a backend to a host's tools. It records tool activity, applies approval policy, and builds the model briefing.

In [ ]:
#| default_exp agent

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail
from ramabana.tools import NullHost
from ramabana.testing import FullHost, MemHost, FakeBackend, fake_agent

In [ ]:
#| export
import datetime, functools, json, re, threading, time, uuid
from dataclasses import dataclass, field
from pathlib import Path
from fastcore.basics import patch
from urai import parse_args, tc_name
from ramabana.core import agent_err, available_models, BranchChanged, budget_for, JOBS, Routing, model_note, tool_channel
from ramabana.runtime import Usage, Run, current_run, run_context, make_backend, Compactor, compact_notebook_context, notices_block
from ramabana.tools import (mime_for, MAX_TOOL_CHARS, NO_SUB, WRITE_TOOLS, Registry, clip, discover,
                            summarise, summary, one_line as _1,
                            err, failed, find, load, read_only, skill_index, subagent_tools,
                            tools_for, Background)
from ramabana.monitor import (Monitors, POB_READER, beat_notes, beat_notice, monitor_tools,
                              pob, pob_path, review_notice)

## The activity feed

The activity feed uses each tool's `summary` function. It does not render generic `name(args)` strings.

In [ ]:
#| export
MAX_DETAIL = 4000     # chars of a tool result kept for the fold
MAX_ACTS = 500        # a very long turn should not grow without bound
RESUME_DETAIL = 600   # chars of a replayed tool result: a resume rebuilds every turn at once
MAX_CHECKPOINTS = 20  # turn boundaries kept for `fork`. Each one is a whole conversation
POLL_EVERY = 900      # seconds between automatic `Host.poll` ticks. A turn is what triggers one
SHELL_SNAPSHOT = 32_000_000

ICONS = {'search': '🔍', 'view': '📄', 'edit': '✏️', 'web': '🌐', 'run': '▶️','skill': '📚', 'delegate': '🤝', 'memory': '🧠', 'watch': '⏰', 'cart': '🛒','tool': '🔧'}

_KIND = {
    'search_code': 'search', 'similar_code': 'search', 'outline': 'search', 'list_files': 'search',
    'grep': 'search', 'ls': 'search',
    'view_file': 'view', 'notebook_cells': 'view', 'view_cell': 'view', 'read_terminal': 'view',
    'edit_file': 'edit', 'replace_text': 'edit', 'create_file': 'edit', 'edit_cell': 'edit',
    'add_cell': 'edit',
    'web_search': 'web', 'read_url': 'web', 'research': 'web',
    'run_python': 'run', 'list_vars': 'run', 'run_shell': 'run',
    'read_skill': 'skill', 'create_skill': 'skill',
    'delegate_search': 'delegate', 'delegate_parallel': 'delegate',
    'inspect_python': 'run',
    'memory_search': 'memory', 'memory_tree': 'memory', 'memory_read': 'memory',
    'memory_topics': 'memory', 'memory_forget': 'memory', 'remember': 'memory',
    'set_reminder': 'watch', 'watch_url': 'watch', 'list_watches': 'watch',
    'cancel_watch': 'watch', 'poll_watches': 'watch',
    'watch_folder': 'watch', 'list_folder_watches': 'watch',
    'cancel_folder_watch': 'watch', 'check_folders': 'watch',
    'cart_stores': 'cart', 'cart_open': 'cart', 'cart_find': 'cart',
    'cart_add': 'cart', 'cart_show': 'cart', 'cart_remove': 'cart',
}
DELEGATE_TOOLS = {t for t, k in _KIND.items() if k == 'delegate'}

In [ ]:
tool = {t.__name__: t for t in tools_for(FullHost())}
[summarise(tool[t], a) for t, a in [
    ('search_code', {'query': 'where is compaction triggered'}),
    ('view_file', {'path': 'nbs/01_runtime.ipynb', 'start': 40, 'end': 80}),
    ('edit_cell', {'path': 'nbs/00_core.ipynb', 'cell_id': '451586c0'}),
    ('inspect_python', {'code': 'list(df.columns)', 'scope': 'overlay'}),
    ('git_status', {}),
]]

['Search where is compaction triggered',
 'View nbs/01_runtime.ipynb:40-80',
 'Edit nbs/00_core.ipynb cell 451586c0',
 'Inspect: list(df.columns)  [overlay]',
 'Delegate 2 questions in parallel: what imports fastllm?; where is threshold used?']

A tool nobody wrote a summary for still gets a usable line, which is what keeps an extension's tool from being invisible in the feed.

In [ ]:
test_eq(summarise(tool['view_file'], {'path': 'a.py', 'start': 4}), 'View a.py:4-')
test_eq(summarise(tool['view_file'], {'path': 'a.py', 'end': 8}), 'View a.py:-8')
test_eq(summarise('view_file', {'path': 'a.py'}), 'View a.py')      # by name, off the index
# every tool an agent is given carries one, so nothing renders as its own call
bare = sorted(t.__name__ for t in tools_for(FullHost()) if getattr(t, 'summary', None) is None)
test_eq(bare, [])
summarise('word_count', {'path': '/proj/a.py'})                     # and one nobody marked says so

"word_count(path='/proj/a.py')"

An `Act` is one call from the moment it starts. It exists *before* the result does, which is the point: a frontend can show `⏳ Web fetch: …` while the fetch is happening.

In [ ]:
#| export
@dataclass
class Act:
    "One tool call, from the moment it starts to whatever it returned."
    tool: str
    args: dict = field(default_factory=dict)
    summary: str = ''
    detail: str = ''
    ok: bool = True
    done: bool = False
    secs: float = 0.0
    started: float = field(default_factory=time.time)
    id: str = field(default_factory=lambda: uuid.uuid4().hex[:12])
    turn_id: str = ''
    revision: int = 0
    branch_id: str = 'main'
    parent_action_id: str = ''
    state: str = 'running'

    @property
    def kind(self): return _KIND.get(self.tool, 'tool')

    @property
    def icon(self): return ICONS.get(self.kind, ICONS['tool'])

    def finish(self, out, ok=True):
        self.detail = _clip(out)
        self.ok, self.done, self.state = ok, True, ('complete' if ok else 'failed')
        self.secs = round(time.time() - self.started, 2)
        return self

    def line(self):
        "The single line: icon, what it did, and how long. Or an hourglass while it runs."
        if not self.done: return f'⏳ {self.summary}'
        tail = f'  ({self.secs:.1f}s)' if self.secs >= 0.5 else ''
        return f'{"" if self.ok else "⚠️ "}{self.icon} {self.summary}{tail}'

    def md(self, fold=True):
        "This call as markdown: the line, with the result folded under it when there is one."
        if not (self.detail and fold): return f'- {self.line()}'
        body = self.detail.replace('```', '`​``')     # a fence in a result must not end ours
        return (f'- {self.line()}\n\n'
                f'  <details><summary>result</summary>\n\n  ```\n{_indent(body)}\n  ```\n\n  </details>')

    def dict(self):
        return {'id': self.id, 'action_id': self.id, 'turn_id': self.turn_id,
                'revision': self.revision, 'branch_id': self.branch_id,
                'parent_action_id': self.parent_action_id, 'state': self.state,
                'tool': self.tool, 'kind': self.kind, 'icon': self.icon,
                'summary': self.summary, 'line': self.line(), 'detail': self.detail,
                'ok': self.ok, 'done': self.done, 'secs': self.secs,
                'args': {k: _1(v, 300) for k, v in (self.args or {}).items()}}


def _clip(out, n=MAX_DETAIL):
    s = '' if out is None else str(out)
    return s if len(s) <= n else s[:n] + f'\n…[{len(s)-n} more chars]'


def _indent(s, pad='  '): return '\n'.join(pad + l for l in s.splitlines())

def _resumed_acts(acts):
    "Persisted tool calls as text, for a context rebuilt from the log rather than a snapshot."
    rows = []
    for a in acts or ():
        if not isinstance(a, dict) or not a.get('tool'): continue
        args = ', '.join(f'{k}={v}' for k, v in (a.get('args') or {}).items())
        row = f"- {a['tool']}({args})" + ('' if a.get('ok', True) else '  [failed]')
        if a.get('detail'): row += '\n' + _indent(_clip(a['detail'], RESUME_DETAIL))
        rows.append(row)
    if not rows: return ''
    return ('<earlier_tool_calls note="replayed from a saved session; arguments and results are truncated">\n'
            + '\n'.join(rows) + '\n</earlier_tool_calls>\n\n')

In [ ]:
a = Act('read_url', {'url': 'https://nbdev.fast.ai/api/export.html'},
        summarise(tool['read_url'], {'url': 'https://nbdev.fast.ai/api/export.html'}))
a.kind, a.icon, a.line()

('web', '🌐', '⏳ Web fetch: https://nbdev.fast.ai/api/export.html')

Finished, it carries the outcome, the duration and the result. `state` moves to `complete` or `failed`, which is what a UI colours on.

In [ ]:
a.finish('# Exporting a notebook to a library\n\nnb_export(...)')
a.line(), a.state

('🌐 Web fetch: https://nbdev.fast.ai/api/export.html', 'complete')

As markdown, the result is folded under the line. An answer buried under thirty tool calls is an answer nobody reads. The working is one click away rather than in the way.

In [ ]:
print(a.md())

- 🌐 Web fetch: https://nbdev.fast.ai/api/export.html

  <details><summary>result</summary>

  ```
  # Exporting a notebook to a library
  
  nb_export(...)
  ```

  </details>


In [ ]:
bad = Act('edit_file', {'path': 'a.py'}, 'Edit a.py').finish('edit failed: stale address', ok=False)
test_eq(bad.state, 'failed')
bad.line()

'⚠️ ✏️ Edit a.py'

`Activity` is the stream, and the hook a frontend hangs a redraw on. `on_change` fires twice per call. Once at the start and once at the end. This is the entire difference between a UI that looks alive and one that looks stuck.

In [ ]:
#| export
class Activity:
    "The stream of calls for a session, and the hook a frontend hangs a redraw on."

    def __init__(self, on_change=None, max_acts=MAX_ACTS):
        self.acts, self.on_change, self.max_acts = [], on_change, max_acts
        self._lock = threading.Lock()
        self._mark = 0
        self.turn_id = ''

    def __len__(self): return len(self.acts)

    def start(self, tool, args, action_id='', turn_id='', revision=0, branch_id='main',
              parent_action_id='', summary=None):
        a = Act(tool=tool, args=dict(args or {}),
                summary=summary if summary is not None else summarise(tool, args),
                id=action_id or uuid.uuid4().hex[:12], turn_id=turn_id or self.turn_id,
                revision=int(revision or 0), branch_id=branch_id or 'main',
                parent_action_id=parent_action_id or '')
        with self._lock:
            self.acts.append(a)
            if len(self.acts) > self.max_acts:
                n = len(self.acts) - self.max_acts
                del self.acts[:n]
                self._mark = max(0, self._mark - n)
        self._changed(a)
        return a

    def finish(self, act, out, ok=True):
        act.finish(out, ok)
        self._changed(act)
        return act

    def _changed(self, act):
        if not self.on_change: return
        try: self.on_change(act)
        except Exception: pass

    def mark(self, turn_id=''):
        "Remember where the stream is now and bind new actions to one durable turn id."
        self._mark = len(self.acts)
        if turn_id: self.turn_id = str(turn_id)
        return self._mark

    def since(self, mark=None):
        return self.acts[(self._mark if mark is None else mark):]

    def rows(self, n=None, mark=None):
        "The stream as dicts, for a frontend. `mark` limits it to one turn."
        acts = self.since(mark) if mark is not None else self.acts
        return [a.dict() for a in (acts[-n:] if n else acts)]

    def md(self, mark=None, fold=True, title='what I did'):
        "The stream as markdown, for saving into a notebook cell."
        acts = self.since(mark) if mark is not None else self.acts
        if not acts: return ''
        body = '\n'.join(a.md(fold) for a in acts)
        return f'<details><summary>{title} ({len(acts)} steps)</summary>\n\n{body}\n\n</details>'

    def lines(self, mark=None):
        "Just the summary lines, for a status pane with no room for folds."
        return [a.line() for a in (self.since(mark) if mark is not None else self.acts)]

In [ ]:
seen = []
feed = Activity(on_change=lambda act: seen.append((act.tool, act.done)))
act = feed.start('search_code', {'query': 'compaction'})
feed.finish(act, 'nbs/01_runtime.ipynb:88  threshold')
seen

[('search_code', False), ('search_code', True)]

`Activity` calls `on_change` outside its lock. Callback exceptions do not interrupt the model turn.

In [ ]:
noisy = Activity(on_change=lambda act: 1/0)
noisy.finish(noisy.start('outline', {'path': 'a.py'}), 'ok')
len(noisy)

1

`mark` remembers where a turn began. The same stream serves the session view and the one-turn fold.

In [ ]:
feed.mark('turn_000002')
feed.finish(feed.start('view_file', {'path': 'a.py'}), 'def a(): return 1')
feed.lines(), feed.lines(mark=feed._mark)

(['🔍 Search compaction', '📄 View a.py'], ['📄 View a.py'])

In [ ]:
test_eq(len(feed.rows()), 2)
test_eq(len(feed.rows(mark=feed._mark)), 1)
feed.rows(mark=feed._mark)[0]['turn_id']

'turn_000002'

## Approvals

Write tools require approval before execution. Each request includes a tool-specific preview. A refusal returns its reason to the model.

In [ ]:
#| export
DENIED = 'Denied by human operator'

DFLT_TIMEOUT = 300      # seconds to wait for a person before giving up on one request
MAX_PREVIEW = 2000      # chars of "what would change". A person will not read more

def _tc(tool_call):
    "Canonical `(name, args)` from a tool call."
    fn = tool_call.get('function') or {}
    return tc_name(tool_call), parse_args(fn.get('arguments'))

In [ ]:
#| export
def _fmt_cmds(commands):
    "exhash commands as one readable block, rather than as a JSON blob nobody reads."
    try:
        cmds = json.loads(commands) if isinstance(commands, str) else commands
        if not isinstance(cmds, list): raise ValueError
    except Exception:
        return str(commands)[:MAX_PREVIEW]
    out = []
    for c in cmds:
        if not isinstance(c, (list, tuple)) or not c: out.append(str(c)); continue
        addr, op, rest = c[0], (c[1] if len(c) > 1 else ''), list(c[2:])
        body = '\n'.join('    ' + str(r).replace('\n', '\n    ') for r in rest)
        out.append(f'{addr} {op}' + (f'\n{body}' if body else ''))
    return '\n'.join(out)


def preview_for(name, args, host=None):
    "What this call would actually do, as text a person can read in a couple of seconds."
    p = args.get('path', '')
    if name == 'edit_file':   return f'{p}\n\n{_fmt_cmds(args.get("commands", ""))}'[:MAX_PREVIEW]
    if name == 'edit_cell':   return f'{p} cell {args.get("cell_id","?")}\n\n{_fmt_cmds(args.get("commands",""))}'[:MAX_PREVIEW]
    if name == 'create_file':
        text, exists = args.get('text', ''), False
        try: exists = bool(host and host.check(p).exists())
        except Exception: pass
        head = f'{p}  ({"OVERWRITES an existing file" if exists else "new file"}, {len(text)} chars)\n\n'
        return (head + text)[:MAX_PREVIEW]
    if name == 'add_cell':
        return f'{p}  (new {args.get("cell_type","code")} cell at {args.get("index",-1)})\n\n{args.get("source","")}'[:MAX_PREVIEW]
    if name == 'run_python':  return str(args.get('code', ''))[:MAX_PREVIEW]
    return json.dumps(args, indent=2, default=str)[:MAX_PREVIEW]


def _summary(name, args):
    "The one-line version, for a status bar or a footer."
    if p := args.get('path'): return f'{name} → {p}'
    return f'{name}({", ".join(sorted(args))})'

Rishi supplies canonical tool-call dictionaries. `_tc` returns the call name and parsed arguments.

In [ ]:
_tc({'function': {'name': 'edit_file', 'arguments': '{"path": "a.py", "commands": "[]"}'}})

('edit_file', {'path': 'a.py', 'commands': '[]'})

The preview is per tool, because the useful preview is different every time: for an edit it is the commands, for a new file it is the head of the file.

In [ ]:
print(preview_for('edit_file', {'path': 'a.py', 'commands': '[["2|ab12|", "s", "return 1", "return 2"]]'}))

a.py

2|ab12| s
    return 1
    return 2


In [ ]:
print(preview_for('create_file', {'path': 'new.py', 'text': 'def f(): pass\n'}, host=NullHost(['/proj'])))

new.py  (new file, 14 chars)

def f(): pass



Anything unrecognised falls back to the arguments, which is still better than a tool name on its own.

In [ ]:
print(preview_for('word_count', {'path': '/proj/a.py'}))

{
  "path": "/proj/a.py"
}


An `Ask` is one request and the person's answer to it. It is truthy exactly when approved. An `Ask` *is* the decision. Both backends' `if not approve(tc)` keeps working, and the reason travels with it.

In [ ]:
#| export
@dataclass
class Ask:
    "One request, and the person's answer to it. `answer` is None while it is pending."
    tool: str
    args: dict = field(default_factory=dict)
    summary: str = ''
    preview: str = ''
    id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])
    answer: bool = None
    note: str = ''
    asked: float = field(default_factory=time.time)
    run_id: str = ''          # the run that raised it; '' when the foreground turn did
    _done: threading.Event = field(default_factory=threading.Event, repr=False, compare=False)

    @property
    def pending(self): return self.answer is None

    def __bool__(self):
        "Truthy exactly when approved. An `Ask` *is* the approval decision."
        return self.answer is True

    def dict(self):
        return {'id': self.id, 'tool': self.tool, 'summary': self.summary, 'preview': self.preview,
                'answer': self.answer, 'note': self.note, 'pending': self.pending,
                'run_id': self.run_id}

    def resolve(self, ok, note=''):
        self.answer, self.note = bool(ok), note or ''
        self._done.set()
        return self

    def wait(self, timeout):
        "Block until answered or `timeout` seconds pass. Returns whether it was answered."
        return self._done.wait(timeout)

    def reply(self):
        "What the model is told: a refusal with its reason, an approval with its note."
        if self.answer: return f'Approved by the user. Note from the user: {self.note}' if self.note else None
        return f'{DENIED}. Reason given: {self.note}' if self.note else DENIED

In [ ]:
#| export
def ask_md(ask):
    "An approval request as markdown. What a person reads, and what is saved in the notebook."
    body = ask.preview.strip()
    fence = '```\n' + body + '\n```\n\n' if body else ''
    return (f'**🔐 approval needed -- `{ask.tool}`**\n\n{ask.summary}\n\n{fence}'
            'Approve, or refuse with a reason -- the reason goes back to the model.')

def answer_md(ask):
    "The person's half of the exchange, in the same voice."
    head = '**✅ approved**' if ask.answer else '**⛔ refused**'
    return f'{head} -- `{ask.tool}`' + (f'\n\n{ask.note}' if ask.note else '')

In [ ]:
ask = Ask('edit_file', {'path': 'a.py'}, 'edit_file → a.py', 'a.py\n\n2|ab12| s')
ask.pending, bool(ask)

(True, False)

A refusal with a reason is the whole point: the model is told why, and can change the approach instead of retrying the same edit.

In [ ]:
ask.resolve(False, 'keep the docstring')
bool(ask), ask.reply()

(False, 'Denied by human operator. Reason given: keep the docstring')

An approval with a note carries it too. "yes, but keep the docstring" is guidance the model should have *while* making the edit.

In [ ]:
test_eq(Ask('edit_file').resolve(True).reply(), None)          # a plain yes needs no words
Ask('edit_file').resolve(True, 'but keep the docstring').reply()

'Approved by the user. Note from the user: but keep the docstring'

In [ ]:
print(ask_md(Ask('create_file', {'path': 'new.py'}, 'create_file → new.py', 'new.py  (new file, 15 chars)')))

**🔐 approval needed -- `create_file`**

create_file → new.py

```
new.py  (new file, 15 chars)
```

Approve, or refuse with a reason -- the reason goes back to the model.


In [ ]:
answer_md(ask)

'**⛔ refused** -- `edit_file`\n\nkeep the docstring'

`Approvals` permits one pending request. A model that requests four writes receives four separate approval decisions.

In [ ]:
#| export
class Approvals:
    "The queue of one, and the thread handshake behind it. One request at a time."

    def __init__(self,
                 tools=(),                  # tool names that need approval. Everything else runs
                 mode='ask',                # 'ask' | 'auto' (approve everything) | 'off' (refuse everything)
                 timeout=DFLT_TIMEOUT,
                 host=None,                 # for previews that need to look at disk
                 on_ask=None,               # called with the `Ask` when one is raised
                 on_answer=None):           # called with the `Ask` when it is answered
        self.tools, self.mode, self.timeout, self.host = frozenset(tools), mode, timeout, host
        # the application's recorder. Frontends register through `listen` instead. Neither unhooks the other
        self.on_ask, self.on_answer = on_ask, on_answer
        self.current = None                 # the `Ask` in flight, or None
        self.closed = False                 # set once; a closing session refuses rather than waits
        self.history = []                   # every `Ask` this session, answered or not
        self._watchers = []                 # (on_ask, on_answer) per registered frontend
        self._lock = threading.Lock()


    @property
    def listeners(self): return len(self._watchers)

    def listen(self, on_ask=None, on_answer=None):
        "Register a frontend, and how to reach it. Returns a callable that unregisters it."
        w = (on_ask, on_answer)
        with self._lock: self._watchers.append(w)
        done = [False]
        def stop():
            if done[0]: return
            done[0] = True
            with self._lock:
                if w in self._watchers: self._watchers.remove(w)
        return stop

    def _notify(self, which, a):
        "Call the recorder and every watcher, swallowing failures so one bad frontend cannot block a turn."
        fns = [getattr(self, f'on_{which}')] + [w[0 if which == 'ask' else 1] for w in list(self._watchers)]
        for f in fns:
            if not f: continue
            try: f(a)
            except Exception: pass

    @property
    def pending(self):
        "The request waiting for an answer, or None. What both frontends poll."
        a = self.current
        return a if (a is not None and a.pending) else None

    def answer(self, id, ok, note='', session=False):
        "Answer the pending request. Optionally approve all later writes this session."
        a = self.current
        if a is None or a.id != id or not a.pending: return None
        if ok and session: self.mode = 'auto'   # only an approval may turn the policy off
        # record and notify before waking the model thread. A recorder finishes first
        a.answer, a.note = bool(ok), note or ''
        if ok and session and not a.note: a.note = 'approved for the rest of this session'
        self._notify('answer', a)
        a._done.set()
        return a

    def close(self):
        "Refuse everything still waiting, and every ask after it. A closing session answers nothing."
        with self._lock:
            if self.closed: return []
            self.closed = True
            # `history` holds them all. `current` is only the newest, and a background run can
            # raise one while another is already waiting
            waiting = [a for a in self.history if a.pending]
        for a in waiting:
            a.resolve(False, 'the session closed before this was answered')
            self._notify('answer', a)
        return waiting

    def _decided(self, a, ok, note):
        "Resolve without asking anybody, and still tell the recorder."
        a.resolve(ok, note)
        self._notify('answer', a)
        return a

    def cancel_all(self, note='the turn was cancelled'):
        "Refuse anything in flight. A stopped turn does not leave a worker thread parked."
        a = self.pending
        if a is not None: self.answer(a.id, False, note)


    def gate(self, tool_call):
        "The `approve(tool_call)` both backends call. Blocks the model's thread."
        return self.request(*_tc(tool_call))

    def request(self, name, args, force=False, timeout=None):
        "Raise one request and wait for it. Returns the resolved `Ask`, whose `reply()` carries the reason."
        a = Ask(tool=name, args=args, summary=_summary(name, args),
                preview=preview_for(name, args, self.host),
                run_id=getattr(current_run(), 'id', '') or '')
        self.history.append(a)
        if not force and name not in self.tools: return a.resolve(True)   # `force` asks anyway
        if self.mode == 'auto': return a.resolve(True)
        if self.mode == 'off': return self._decided(a, False, 'approval is switched off for this session')
        # closing first: it is the more useful reason, and it holds whether or not anyone listens.
        # `current` is taken under the same lock a close competes for, because checking and then
        # storing separately left an ask that landed in the gap waiting out its whole timeout
        with self._lock:
            closing = self.closed
            if not closing: self.current = a
        if closing: return self._decided(a, False, 'the session is closing')
        if self.listeners < 1:
            return self._decided(a, False, 'nothing is listening for approvals, so this could not be asked')
        self._notify('ask', a)
        wait_for = self.timeout if timeout is None else timeout
        if not a.wait(wait_for):
            a.resolve(False, f'no answer after {wait_for}s')
            self._notify('answer', a)
        return a

In [ ]:
gate = Approvals(tools={'edit_file'}, mode='ask', timeout=1)
test_eq(gate.request('edit_file', {'path': '/proj/a.py'}).note,
        'nothing is listening for approvals, so this could not be asked')   # refused, not waited out

# an ask records which run raised it, so a frontend can say who is asking
from ramabana.runtime import Run, run_context
auto = Approvals(tools={'edit_file'}, mode='auto')
with run_context(Run('run_bg', 'background', 'q')):
    test_eq(auto.request('edit_file', {}).run_id, 'run_bg')
test_eq(auto.request('edit_file', {}).run_id, '')          # the foreground turn claims no run

test_eq(gate.closed, False)
test_eq(gate.close(), [])                                  # nothing was waiting
test_eq(gate.closed, True)
test_eq(gate.request('edit_file', {}).note, 'the session is closing')

With nothing listening it refuses immediately and says so. Otherwise `gate` would park the model's worker thread until the timeout waiting for an answer that was never going to come.

In [ ]:
gate = Approvals(tools=WRITE_TOOLS)
gate.listeners, gate.request('edit_file', {'path': 'a.py'}).reply()

(0,
 'Denied by human operator. Reason given: nothing is listening for approvals, so this could not be asked')

A frontend registers, and answers. Here it answers inside the notification, which is exactly what a UI does a moment later on a human's behalf.

In [ ]:
stop = gate.listen(on_ask=lambda a: gate.answer(a.id, False, 'rewrite it as a patch instead'))
decision = gate.request('edit_file', {'path': 'a.py', 'commands': '[]'})
bool(decision), decision.reply()

(False,
 'Denied by human operator. Reason given: rewrite it as a patch instead')

Tools outside the approval policy run without an approval request.

In [ ]:
test_eq(bool(gate.request('search_code', {'query': 'x'})), True)
test_eq(gate.pending, None)
len(gate.history)

3

The two bulk answers are deliberate acts rather than a slip of the return key: `'auto'` approves everything for the session, `'off'` refuses everything and says which.

In [ ]:
Approvals(tools=WRITE_TOOLS, mode='auto').request('edit_file', {}).reply(), Approvals(tools=WRITE_TOOLS, mode='off').request('edit_file', {}).reply()

(None,
 'Denied by human operator. Reason given: approval is switched off for this session')

`session=True` on an approval is what turns the policy off, and it records that it did.

In [ ]:
stop()
gate.listen(on_ask=lambda a: gate.answer(a.id, True, session=True))
first = gate.request('edit_file', {'path': 'a.py'})
gate.mode, first.note

('auto', 'approved for the rest of this session')

A timeout is a refusal that explains itself, and a cancelled turn releases whoever was waiting rather than leaving a worker thread parked.

In [ ]:
slow = Approvals(tools=WRITE_TOOLS, timeout=0.01)
slow.listen(on_ask=lambda a: None)
slow.request('edit_file', {'path': 'a.py'}).reply()

'Denied by human operator. Reason given: no answer after 0.01s'

`gate` is the `approve(tool_call)` both backends call, and `policy` builds one from per-tool modes. Written out here rather than imported from rishi, because the harness needs one approval shape across both backends and must not stop working because the local engine is not installed.

In [ ]:
#| export
def always(tool_call): return True
def never(tool_call): return False

def policy(modes, ask):
    "`approve(tool_call)` from per-tool modes: 'approved' | 'check' | 'dont_run'."
    def approve(tc):
        name, _ = _tc(tc)
        mode = (modes or {}).get(name, 'check')
        return True if mode == 'approved' else False if mode == 'dont_run' else ask(tc)
    return approve

In [ ]:
tc = {'function': {'name': 'edit_file', 'arguments': {'path': 'a.py'}}}
bool(gate.gate(tc))

True

In [ ]:
ask_all = Approvals(tools=WRITE_TOOLS, mode='auto').gate
approve = policy({'search_code': 'approved', 'run_python': 'dont_run'}, ask_all)
[bool(approve({'function': {'name': n, 'arguments': {}}})) for n in ('search_code', 'run_python', 'edit_file')]

[True, False, True]

## The fastllm approval shim

Hosted models reach approvals through rishi's own remote path now. This module is three functions reporting that there is nothing left to patch. It stays because callers ask.

In [ ]:
#| export
def applied(): return True
def apply(): return True
def note(): return 'provided by rishi.remote'

In [ ]:
applied(), note()

(True, 'provided by rishi.remote')

## Routing a turn before the model sees it

A small deterministic step in front of every turn. Deliberately not another model call: routing "what default model does leela use?" to the web is exactly the mistake this prevents, and tool choice should be predictable enough to read in the history.

In [ ]:
#| export
INLINE_SKILLS = ('exhash', 'coding_patterns')


def tool_plan(prompt):
    "A small deterministic routing step before the model sees a turn. Never another model call."
    p = str(prompt or '').lower()
    repo = ('this repo', 'repository', 'codebase', 'implementation', 'implemented',
            'default model', 'config', 'source code', 'where is', 'which file', ' method',
            ' function', ' class')
    current = ('latest', 'current docs', 'documentation says', 'release notes', 'on the web',
               'today', 'recent version')
    action = ('create', 'make', 'scale', 'run', 'execute', 'fix', 'change', 'add ', 'remove', 'rename')
    if any(x in p for x in repo):
        return ('repo', 'Use search_code first. Read the matching source if needed. '
                        'Do not web-search a question about the open repository.')
    if any(x in p for x in current):
        return ('web', 'Use web_search, then read_url for the authoritative result.')
    if p.strip().startswith(action) or ' as df_' in p:
        return ('act', 'Use the execution/editing tool that produces the requested result, then verify it.')
    return ('direct', 'Answer directly; use a tool only if the available context is insufficient.')

In [ ]:
[(p, tool_plan(p)[0]) for p in ('where is compaction triggered?',
                                 'what do the latest nbdev release notes say?',
                                 'create a test for threshold',
                                 'why is 2+2 four?')]

[('where is compaction triggered?', 'repo'),
 ('what do the latest nbdev release notes say?', 'web'),
 ('create a test for threshold', 'act'),
 ('why is 2+2 four?', 'direct')]

In [ ]:
test_eq(tool_plan('which file defines Routing?')[0], 'repo')
tool_plan('which file defines Routing?')[1]

'Use search_code first. Read the matching source if needed. Do not web-search a question about Leela or the open repository.'

The frontend composes a message around the question. The open notebook, the screen. `request_text` recovers the part the person actually typed. Everything that inspects intent looks at that, not at the envelope.

In [ ]:
#| export
def request_text(prompt):
    """The person's request, excluding notebook/screen context composed around it."""
    text = prompt[-1] if isinstance(prompt, (list, tuple)) and prompt else prompt
    text = str(text or '')
    matches = list(re.finditer(r'<user-request>\n?(.*?)\n?</user-request>', text, re.S))
    return matches[-1].group(1) if matches else text

In [ ]:
request_text('<notebook path=a.ipynb>\n...\n</notebook>\n\n<user-request>\nscale df as df_norm\n</user-request>')

'scale df as df_norm'

A prompt may name a tool or a skill outright. Only names already exposed to the agent count. A URL or a path is left alone.

In [ ]:
#| export
def prompt_directives(prompt, tools=(), skills=()):
    "Explicit `/tool` and `/skill-name` mentions in an ordinary prompt. Known names only."
    text = str(prompt or '')
    tool_names = {getattr(t, '__name__', ''): t for t in tools}
    skill_names = {s.name.lower(): s for s in skills}
    matches = list(re.finditer(r'(?<!\S)/([A-Za-z_][\w-]*)', text))
    requested, loaded = [], []
    for i, match in enumerate(matches):
        name = match.group(1)
        if name in tool_names:
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
            requested.append((name, text[match.end():end].strip()))   # the text after it is the suggested query
        elif name.lower() in skill_names:
            loaded.append(skill_names[name.lower()])
    return requested, loaded

In [ ]:
host = MemHost({'/proj/a.py': 'def a(): return 1\n'})
ts = tools_for(host)
prompt_directives('/search_code compaction threshold  and see https://x/y/z', tools=ts)

([('search_code', 'compaction threshold  and see https://x/y/z')], [])

## The briefing

The system prompt: what the agent is, where it is, how to work, and what it knows. The skill index is names and descriptions only. Bodies are a `read_skill` away. Because a dozen full skill texts would crowd out the code the model is meant to be looking at.

In [ ]:
#| export
MAX_CONTEXT_FILE = 8000     # chars of one AGENTS.md. Past this it is documentation, not instructions
CONTEXT_FILES = ('AGENTS.md', '.agents/AGENTS.md', '.leela/AGENTS.md')

def project_context(host, mx=MAX_CONTEXT_FILE):
    "The project's own instructions to an agent, from `AGENTS.md` in each open folder."
    from pathlib import Path
    out, seen = [], set()
    for r in host.roots or ():
        for rel in CONTEXT_FILES:
            try: p = host.check(Path(r)/rel)
            except Exception: continue
            if str(p) in seen: continue
            seen.add(str(p))
            try: text = host.read(str(p))
            except Exception: text = None
            if not text or not text.strip(): continue
            body = text.strip()
            if len(body) > mx: body = body[:mx] + f'\n…[truncated; read {p} in full if you need the rest]'
            out.append(f'<project_instructions path="{p}">\n{body}\n</project_instructions>')
    if not out: return ''
    return ('\n\n<project_context>\nInstructions this project gives to any agent working in it. They\n'
            'override the general guidance above where they disagree.\n\n' + '\n\n'.join(out) +
            '\n</project_context>')

RULES = (
    (None, 'Act on the user’s verb. “Create”, “run”, “fix”, “add” and “as NAME” request a\n'
           '  result, not a plan: use the tool that produces it, verify it, then report what exists.\n'
           '  Never stop at “I will…”.'),
    (None, 'Start every user-facing response with what you plan to do or the next action. Keep the user\n'
           '  informed during ongoing work, and end the response with the result, conclusion, or what is needed.'),
    (None, 'Never claim a file changed, a command passed, or a test went green unless a tool\n'
           '  result in this conversation says so. If you did not run it, say you did not run it.'),
    (None, 'A tool result starting with ERROR: is a failure. Read it, fix the cause, and try a\n'
           '  different approach. Calling the same tool again unchanged is never the fix.'),
    (None, 'Keep a plan small enough that every step has one independently verifiable outcome. Do not\n'
           '  begin the next step until the current step is verified. If verification would widen the scope,\n'
           '  split the step or stop and report the blocker.'),
    ('search_code', 'Use `search_code` for project behaviour, unfamiliar APIs, or uncertainty about an\n'
                    '  installed library -- the index covers this repo *and* every installed package. Do not\n'
                    '  search for routine Python you already know.'),
    ('grep', 'Use `grep` when you know the exact string: a symbol you are renaming, an error\n'
             '  message, an import. `search_code` finds what is *like* the query; `grep` finds every\n'
             '  place it literally occurs, which is what a rename or an audit needs.'),
    ('view_file', 'Prefer `view_file` over guessing. Copy paths exactly from the workspace, notebook\n'
                  '  context, or search results; never shorten, repair, or reconstruct a path.'),
    ('replace_text', 'To change a file: read it, then `replace_text` with `oldText` copied exactly from\n'
                     '  what you read. Send every change to one file as one call.'),
    ('edit_file', '`edit_file` is the other editor: it addresses lines by the hashes `view_file`\n'
                  '  returns, so a stale read fails instead of damaging the wrong line. Use it when that\n'
                  '  matters, and read the `exhash` skill before your first one.'),
    ('notebook_cells', 'Never use `view_file` or `edit_file` on an `.ipynb` file. Call `notebook_cells` on\n'
                       '  its exact path, choose the returned cell id, then `view_cell` and `edit_cell`.'),
    (None, 'A `<notebook path="…">` block is the open notebook and its exact path. Its cells are\n'
           '  already visible; use that path with the notebook tools and never reconstruct it.'),
    ('run_shell', 'Check your work with `run_shell`: after an edit run the project’s tests, after a\n'
                  '  signature change run its linter or type checker. Use the commands the project itself\n'
                  '  documents (README, pyproject, Makefile). Never start a server, watcher or REPL --\n'
                  '  only commands that exit on their own.'),
    ('run_python', 'Code cells inside `<notebook>` have already executed. Their printed `<output>` is not\n'
                   '  Python and must never be copied into `run_python`. For a request about `df`, call\n'
                   '  `list_vars` first, then run only the transformation the user asked for.'),
    ('run_python', '`run_python` shares the user’s kernel namespace. Read anything; bind results to NEW\n'
                   '  names. You cannot rebind or delete the user’s variables, so do not try.'),
    ('web_search', 'Use `web_search`/`read_url` only when the answer depends on current external\n'
                   '  documentation -- not for questions about this repository.'),
    ('memory_search', 'Before acting on a request, search Vishalakshi durable memory with `memory_search` when\n'
                      '  that tool is available; use stored preferences and relevant prior context.'),
    ('read_skill', 'Before writing prose that ships with the work -- a docstring, a comment, a README, a\n'
                   '  commit message, a PR description, a message to a colleague -- read the `write_docs`\n'
                   '  skill. For narrative writing read `write_prose`, and for the design a codebase is\n'
                   '  derived from, `theory`.'),
    ('delegate_parallel', 'When two or more questions are independent and each would take several tool calls,\n'
                          '  send them together with `delegate_parallel` rather than working through them yourself.'),
    ('watch_folder', '`watch_folder` is for work happening beside this conversation: another agent editing\n'
                     '  the repo, a build writing output. Its `instructions` are the whole brief the reviewer\n'
                     '  gets, so write them self-contained. Its reviews arrive on their own; `check_folders`\n'
                     '  looks now.'),
    (None, 'Make the change the user asked for and no other. Do not reformat, reorganise, or\n'
           '  “improve” code you were not asked to touch, and never discard their edits.'),
    (None, 'Writes may be put to the user for approval. A refusal comes back with their reason --\n'
           '  read it and change the approach, do not retry the same call.'),
    (None, 'Write, edit and run only inside the folders above. Anything else is refused.'),
    (None, 'Be concise. Report what you did and what it cost, not what you intend to do.'),
    (None, 'Answer in plain sentences. Headings, bullet lists and bold belong in a document the\n'
           '  user asked for, not in a reply, and a code fence holds code rather than prose.\n'
           '  Formatting a two-line answer as a report is the most common way this briefing is ignored.'),
)


#: Re-asserted after the tag block on the tags channel. Rishi appends the tool protocol *after*
#: the briefing there, so the last thing those models read is tool punctuation rather than the
#: rules. Riding out with the turn is the only way the rules get to be last instead.
OUTPUT_CONTRACT = ('\n\n<output-contract>Reply in plain sentences: no headings, no bullet list, no '
                   'bold, no code fence around prose. Lead with the answer and stop. This outranks any '
                   'formatting habit carried in from another harness.</output-contract>')


def work_rules(names=()):
    "The briefing's rules, keeping those whose tool is on the table. Empty `names` filters nothing."
    names = set(names or ())
    return '\n'.join(f'- {text}' for tool, text in RULES if not names or tool is None or tool in names)


def system_prompt(host, skills=(), inline=INLINE_SKILLS, extra='', tools=()):
    "The agent's briefing: what it is, where it is, how to work, and what it knows."
    names = {getattr(t, '__name__', '') for t in tools or ()}
    roots = '\n'.join(f'  {r}' for r in host.roots) or '  (no folder open)'
    if getattr(host, 'read_outside', False):   # only when the host says so
        roots += ('\n  Reads may name any path on this machine. Writing, running commands and\n'
                  '  listing files stay inside the folders above.')
    # Claimed only where it is true: a host says so, and "keep it short, the kernel is busy"
    # is advice for a problem it may not have.
    conc = ('\n  Your kernel runs each inspection in its own subshell. This works while one '
            "of the user's cells is still running." if getattr(host, 'concurrent', False) else '')
    live = ('' if 'inspect_python' not in names and names else
            '\n- To *look at* live state, prefer `inspect_python`: neither of its scopes can change\n'
            '  what the user made, so it needs no approval. Start with the default sandbox and pass\n'
            f"  `scope='overlay'` when it refuses a library call you need.{conc}")
    sp = f"""You are Ramabana, a coding agent. Follow the user's latest explicit request and the project instructions below. You are working in these folders:
{roots}

You can search the code index (this repo *and* every installed package), read and edit
files, run commands in the project, run Python in the user's live kernel namespace, read
the web, and read skills that describe the tools already installed here.

How to work:
{work_rules(names)}{live}"""
    idx = skill_index(skills)
    if idx: sp += idx
    for name in inline or ():
        if (s := find(skills, name)): sp += f'\n\n## {s.name}\n\n{s.text()}'
    sp += project_context(host)
    return sp + (f'\n\n{extra}' if extra else '')

What the open folders say about themselves: `AGENTS.md` and its neighbours, clipped. This is how a
repository briefs an agent without anyone configuring anything.

In [ ]:
print(project_context(MemHost({'/proj/AGENTS.md': '# House rules\n\nDense lines. Short names.\n'}))[:200])

In [ ]:
got = project_context(MemHost({'/proj/AGENTS.md': '# House rules\n\nDense lines. Short names.\n'}))
assert 'Dense lines.' in got and '<project_context>' in got
test_eq(project_context(MemHost({})), '')                  # nothing to say, so nothing is said
big = project_context(MemHost({'/proj/AGENTS.md': 'x' * (MAX_CONTEXT_FILE * 3)}))
assert len(big) < MAX_CONTEXT_FILE * 2, len(big)           # one file cannot flood the briefing

The rules block in every briefing. Naming a subset is how a small model gets a shorter one.

In [ ]:
print(work_rules()[:280])

In [ ]:
whole = work_rules()
assert whole and isinstance(whole, str)
assert len(work_rules(('evidence',))) < len(whole)         # a subset is shorter than all of them
assert len(work_rules(())) <= len(whole)
assert work_rules() == whole                               # and asking twice says the same thing

In [ ]:
sp = system_prompt(host)
print(sp[:320])

You are Ramabana, a coding agent. Follow the user's latest explicit request and the project instructions below. You are working in these folders:
  /proj

You can search the code index (this repo *and* every installed package), read and edit
files, run commands in the project, run Python in the user's live kernel names


The folders are in it, because a path outside them is refused and the model needs to know which ones those are. With no folder open it says so rather than leaving the line blank.

In [ ]:
test_eq('/proj' in sp, True)
test_eq('(no folder open)' in system_prompt(NullHost()), True)
len(sp)

4204

A Claude model behind the agent drops a particular set of habits, and none of them are about tools. None of them are in `RULES`. `CLAUDE_NOTES` carries them: they are the Answer.AI house behaviours from `aai-coding`'s `sysp.md` and `core.md` that the rules above do not reach. What the environment and git belong to, how far an approval travels, and how a turn ends. Pass it as `extra` when the model is a Claude one. `system_prompt` already appends `extra` last.

In [ ]:
#| export
CLAUDE_NOTES = """## Working as Claude

These apply on top of the rules above, and outrank them where they disagree.

- The environment and the repository history are the user's. Do not install packages, change environment configuration, or run any git command -- read-only ones included -- unless the user approved that exact command in this conversation. Where git would answer something, name the command and ask.
- Approval does not travel. Confirm before an action that is hard to reverse or that is visible outside this machine, and look at the target before you delete or overwrite it. If what you find contradicts how it was described, say that instead of proceeding.
- There is no momentum. Never extend agreed work into new decisions, and when in doubt whether something was agreed, it was not. Approval for a downstream change does not cover an upstream one.
- A question outranks the work in flight. Answer it in prose and end the turn. A question is never approval to continue and never an occasion to change code.
- Never end a response by asking what to do next. Stating a recommendation or naming what remains undone is right; soliciting the next instruction takes agency from the user. Asking for their read on a direction is welcome; asking permission to proceed is not.
- Do not work around a problem. Fix it at its source, or say what is blocking and stop. A broken tool comes before the work in flight, because every later task pays for it.
- Correct the record. When an earlier claim of yours turns out to be wrong, say so plainly rather than moving quietly past it.
- Before finalizing a turn, reflect on mistakes made during it. For each concrete mistake with a reusable correction, record the mistake and its fix in Vishalakshi with `remember`, so later work can avoid it.

- Everything the user needs is in the final text of the turn, with no tool call after it. Text between tool calls may never reach them, so restate anything important that appeared only mid-run.
- Lead with the outcome when the turn concludes: the first sentence says what happened or what you found. Keep the plan-first opener for a turn that will carry on working.
- No metadiscourse. Do not advertise the content ("the key point is", "what's interesting is"), and never end on a caveat or a note. A risk that could change the decision belongs in the body, beside the reasoning it affects.
- Never hard-wrap prose: one paragraph is one line, and the display wraps it. To show markdown the user can copy, use a four-space indented block rather than a fence.
"""

In [ ]:
test_eq(CLAUDE_NOTES in system_prompt(host, extra=CLAUDE_NOTES), True)
test_eq(CLAUDE_NOTES in system_prompt(host), False)

## The session plan

A durable checklist for stop/start and for work a sub-agent can take a bite of. The plan lives on `Agent`, not on the host: CLI, MCP and leela all see the same object, and a cancelled turn does not throw the list away.

Statuses are `pending`, `active`, `done` and `cancelled`. At most one todo is `active`. On resume, continue from the active item rather than restarting the title.

In [ ]:
#| export
TODO_STATUSES = ('pending', 'active', 'done', 'cancelled')
TODO_MARK = {'pending': '[ ]', 'active': '[▸]', 'done': '[x]', 'cancelled': '[-]'}

@dataclass
class Todo:
    "One step on a session plan."
    id: str
    text: str
    status: str = 'pending'   # pending | active | done | cancelled
    note: str = ''
    owner: str = ''           # '' = main agent. A label when a sub-agent owns it

    def __post_init__(self):
        if self.status not in TODO_STATUSES:
            raise ValueError(f'status must be one of {TODO_STATUSES}')

    def dict(self):
        return dict(id=self.id, text=self.text, status=self.status, note=self.note, owner=self.owner)

    @classmethod
    def from_dict(cls, d):
        if isinstance(d, Todo): return d
        if not isinstance(d, dict): return cls(id=uuid.uuid4().hex[:8], text=str(d))
        return cls(id=str(d.get('id') or uuid.uuid4().hex[:8]),
                   text=str(d.get('text') or ''),
                   status=str(d.get('status') or 'pending'),
                   note=str(d.get('note') or ''),
                   owner=str(d.get('owner') or ''))


class Plan:
    "The session's working list: what to do, what's done, what to resume after a stop."
    def __init__(self, title='', todos=None, updated=0.):
        self.title = str(title or '')
        self.todos, self.updated = [], float(updated or time.time())
        for it in todos or ():
            if isinstance(it, Todo): self.todos.append(it)
            elif isinstance(it, dict): self.todos.append(Todo.from_dict(it))
            else:
                text = str(it).strip()
                if text: self.todos.append(Todo(id=uuid.uuid4().hex[:8], text=text))

    def __bool__(self): return bool(self.title) or bool(self.todos)
    def __len__(self): return len(self.todos)

    def dict(self):
        return dict(title=self.title, updated=self.updated, todos=[t.dict() for t in self.todos])

    @classmethod
    def from_dict(cls, d):
        d = dict(d or {})
        return cls(title=d.get('title') or '', todos=d.get('todos') or [], updated=d.get('updated') or 0.)

    def progress(self):
        " `(done, total)` counting cancelled out of the total."
        alive = [t for t in self.todos if t.status != 'cancelled']
        return sum(t.status == 'done' for t in alive), len(alive)

    def active(self):
        return next((t for t in self.todos if t.status == 'active'), None)

    def pending(self):
        return [t for t in self.todos if t.status == 'pending']

    def find(self, key):
        "Exact id, or a unique prefix / unique text substring."
        key = str(key or '').strip()
        if not key: return None
        for t in self.todos:
            if t.id == key: return t
        pref = [t for t in self.todos if t.id.startswith(key)]
        if len(pref) == 1: return pref[0]
        text = [t for t in self.todos if key.lower() in t.text.lower()]
        return text[0] if len(text) == 1 else None

    def _touch(self): self.updated = time.time(); return self

    def clear(self):
        self.title, self.todos = '', []
        return self._touch()

    def set(self, title, items=()):
        "Replace the plan. `items` are todo texts (str) or dicts/`Todo`s."
        self.title = str(title or '').strip()
        out = []
        for it in items or ():
            if isinstance(it, Todo): out.append(it)
            elif isinstance(it, dict): out.append(Todo.from_dict(it))
            else:
                text = str(it).strip()
                if text: out.append(Todo(id=uuid.uuid4().hex[:8], text=text))
        self.todos = out
        return self._touch()

    def add(self, text, status='pending', owner='', id=None):
        t = Todo(id=id or uuid.uuid4().hex[:8], text=str(text).strip(), status=status, owner=owner)
        if not t.text: raise ValueError('todo text is empty')
        self.todos.append(t)
        return self._touch() and t

    def update(self, key, status=None, note=None, text=None, owner=None):
        t = self.find(key)
        if t is None: raise KeyError(f'no todo matches {key!r}')
        if status is not None:
            if status not in TODO_STATUSES: raise ValueError(f'status must be one of {TODO_STATUSES}')
            if status == 'active':
                for o in self.todos:
                    if o is not t and o.status == 'active': o.status = 'pending'
            t.status = status
        if note is not None: t.note = str(note)
        if text is not None: t.text = str(text).strip() or t.text
        if owner is not None: t.owner = str(owner)
        self._touch()
        return t

    def md(self):
        "Checklist for a briefing, a status block, or a `/plan` reply."
        if not self: return '(no plan)'
        done, total = self.progress()
        head = self.title or 'Plan'
        lines = [f'**{head}**  ·  {done}/{total} done']
        for t in self.todos:
            mark = TODO_MARK.get(t.status, '[ ]')
            own = f'  @{t.owner}' if t.owner else ''
            note = f'  -- {t.note}' if t.note else ''
            lines.append(f'{mark} `{t.id}` {t.text}{own}{note}')
        return '\n'.join(lines)

    def line(self):
        "One status-bar fragment, or '' when empty."
        if not self: return ''
        done, total = self.progress()
        cur = self.active()
        bit = f'{done}/{total}'
        if cur: bit += f' ▸ {cur.text[:40]}'
        elif self.pending(): bit += f' · next: {self.pending()[0].text[:36]}'
        return bit


def parse_plan_items(items):
    "Newline text, a JSON list, or a Python list -> todo texts."
    if items is None or items == '': return []
    if isinstance(items, (list, tuple)): return [str(x).strip() for x in items if str(x).strip()]
    s = str(items).strip()
    if s.startswith('['):
        try:
            data = json.loads(s)
            if isinstance(data, list): return [str(x).strip() for x in data if str(x).strip()]
        except Exception: pass
    return [ln.strip().lstrip('-* ').strip() for ln in s.splitlines() if ln.strip()]


def plan_tools(get_plan, save=None):
    """Model-facing plan tools. Closures over the agent's `Plan` so Host stays free of them."""
    def _save():
        if save:
            try: save()
            except Exception: pass

    @summary(lambda a: f'Set plan: {_1(a.get("title"))}')
    def set_plan(title: str, items: str = '') -> str:
        "Replace the session plan. `items` is a newline list or a JSON list of step texts."
        todos = parse_plan_items(items)
        if not str(title or '').strip() and not todos:
            get_plan().clear(); _save(); return 'plan cleared'
        get_plan().set(title, todos); _save()
        return get_plan().md()

    @summary(lambda a: f'Add todo: {_1(a.get("text"))}')
    def add_todo(text: str, status: str = 'pending') -> str:
        "Append one step to the session plan."
        try: t = get_plan().add(text, status=status or 'pending')
        except Exception as e: return err('could not add todo', e)
        _save()
        return f'added `{t.id}` ({t.status}): {t.text}\n\n{get_plan().md()}'

    @summary(lambda a: f'Todo {a.get("id","?")} → {a.get("status") or "update"}')
    def update_todo(id: str, status: str = '', note: str = '', text: str = '') -> str:
        """Update a todo by id or unique prefix. Status: pending, active, done, cancelled.

        Mark the step you are working on `active`, and `done` when it is finished. After a
        stop, resume from the active step rather than rewriting the plan.
        """
        kw = {}
        if status: kw['status'] = status
        if note != '': kw['note'] = note
        if text: kw['text'] = text
        if not kw: return err('nothing to update; pass status, note or text')
        try: t = get_plan().update(id, **kw)
        except Exception as e: return err('could not update todo', e)
        _save()
        return f'`{t.id}` → {t.status}: {t.text}' + (f' -- {t.note}' if t.note else '') + f'\n\n{get_plan().md()}'

    @summary(lambda a: 'List plan')
    def list_plan() -> str:
        "The current session plan as a checklist."
        return get_plan().md()

    return [set_plan, add_todo, update_todo, list_plan]

A plan arrives from a model as JSON, as lines, or already as a list. All three are unambiguous, so
all three are taken.

In [ ]:
parse_plan_items('["Cut shalya", "Rewire ramabana"]'), parse_plan_items('one\ntwo')

In [ ]:
test_eq(parse_plan_items('["a", "b"]'), ['a', 'b'])
test_eq(parse_plan_items('one\ntwo'), ['one', 'two'])
test_eq(parse_plan_items(['a', 'b']), ['a', 'b'])
test_eq(parse_plan_items(''), [])
test_eq(parse_plan_items('  spaced  '), ['spaced'])

The session plan as tools. A plan is the agent's own record of what it is doing, and both frontends
read it rather than keeping one each. `plan_tools` takes a getter and a saver, so the plan can live
wherever the caller keeps it.

The saver is called with no arguments and its failures are swallowed. Failing to write a plan file
is not a reason to end someone's turn.

In [ ]:
held = Plan('Ship the split', ['Cut shalya', 'Rewire ramabana'])
saves = []
pt = {t.__name__: t for t in plan_tools(lambda: held, save=lambda: saves.append(held.md()))}
sorted(pt)

In [ ]:
test_eq(sorted(pt), ['add_todo', 'list_plan', 'set_plan', 'update_todo'])
assert 'Cut shalya' in pt['list_plan']()
pt['add_todo']('Migrate leela')
assert 'Migrate leela' in pt['list_plan']()
test_eq(len(held.todos), 3)
# a todo is addressed by its own id, not by its position: a plan is reordered, positions are not
pt['update_todo'](held.todos[0].id, 'done')
test_eq(held.todos[0].status, 'done')
assert failed(pt['update_todo']('99', 'done'))             # no such todo, and it says so
assert saves, 'every change is handed to the saver'
pt['set_plan']('A different goal', '["only step"]')
test_eq(held.title, 'A different goal')
test_eq([t.text for t in held.todos], ['only step'])

# a saver that raises is swallowed: failing to write a plan must not end someone's turn
broken = {t.__name__: t for t in plan_tools(lambda: held, save=lambda: 1/0)}
assert 'Recovered' in broken['set_plan']('Recovered', '["still works"]')

In [ ]:
#| hide
p = Plan('Ship the plan feature', ['Design API', 'Wire tools', 'CLI surface', 'Tests'])
test_eq(p.progress(), (0, 4))
p.update(p.todos[0].id, status='active')
test_eq(p.active().text, 'Design API')
p.update('Design', status='done')
test_eq(p.progress(), (1, 4))
assert 'Design API' in p.md() and '[x]' in p.md()
assert not Plan()

## The agent

One `Agent` owns one continuous model context, and everything else hangs off it: routing, tools, skills, extensions, approvals, compaction, the activity feed and the durable history. Availability is reported rather than raised. The model is a multi-gigabyte download on one side and an API key on the other, and an editor that will not open without either is a worse editor.

In [ ]:
#| export
class Agent:
    "The IDE's agent: a routed chat whose tools are the host's own capabilities."

    def __init__(self,
                 host,
                 model=None,                # the turn model. None takes the routing default
                 routing=None,
                 sp=None,                   # override the whole briefing
                 approvals=None,            # an `Approvals`. None means nothing is gated
                 cfg=None,                  # config dir, for skills and extensions
                 compact=True,              # compact automatically at the threshold
                 compact_strategy='summary', # 'summary' model checkpoint | 'surgical' deterministic DSL
                 kernel_alive=True,         # what the post-compaction note may promise
                 extensions=True,
                 project_extensions=False,  # project extensions execute repo code: opt in
                 ext_paths=(),
                 inline_skills=INLINE_SKILLS,
                 subagents=True,
                 subagent_writes=False,     # sub-agents get the write tools too, behind the same approvals
                 readonly=False,            # withhold every tool that acts: this agent may only propose
                 readonly_calls=None,       # and, when set, a hard budget on how many it may make
                 local_multimodal=False,       # load LiteRT vision/audio encoders for local models
                 tool_max_len=MAX_TOOL_CHARS,
                 on_compact=None,
                 on_activity=None,
                 history_name='agent',      # separate durable conversations can share one config dir
                 poll_every=POLL_EVERY,     # seconds between automatic watch polls; 0 never polls
                 instruction_style='ramabana'): # 'ramabana' | 'aai' compatibility profile
        self.host, self.cfg, self.inline_skills = host, cfg, inline_skills
        if instruction_style not in ('ramabana', 'aai'): raise ValueError('instruction_style must be ramabana or aai')
        self.instruction_style = instruction_style
        self.history_name = history_name
        self.session_id = f'agent_{datetime.datetime.now().strftime("%Y%m%d-%H%M%S-%f")}'
        self.turn_seq, self.current_turn_id = 0, ''
        self.current_branch_id, self.checkpoints, self._branch_hist = 'main', {}, {}
        self.routing = routing or Routing(turn=model)
        if model: self.routing.set(model)
        self.approvals, self.tool_max_len, self.subagents = approvals, tool_max_len, subagents
        self.subagent_writes = bool(subagent_writes)
        self.readonly, self.readonly_calls = bool(readonly), readonly_calls
        self.local_multimodal = bool(local_multimodal)
        self.extensions, self.project_extensions, self.ext_paths = extensions, project_extensions, ext_paths
        self._sp = sp
        self.compactor = Compactor(auto=compact, strategy=compact_strategy, kernel_alive=kernel_alive, on_compact=on_compact)
        self.activity = Activity(on_change=on_activity)   # the live account of what it is doing
        self._nested = threading.local()   # per thread: the delegate calls whose sub-agents are running
        self.plan = Plan()       # durable checklist for stop/start and sub-agent bites
        self.on_plan = None      # frontend hook: callable(plan) after every mutation
        self.on_media = None     # frontend hook: callable(paths) the moment a tool writes a picture
        self.calls = []          # (tool, args) per call this session. What the UI shows as activity
        self.history = []        # inspectable user/assistant turns, including the chosen tool plan
        self._load_history()
        self._load_plan()
        self.before = {}         # path -> its text just before a write tool touched it, this turn
        self._walked = False     # whether a tree baseline is held for the running command
        self._tree = {}          # that baseline: path -> its text when the command started
        self._tool_calls_turn = 0 # backend-independent guard for local/native tool loops
        self.max_tool_calls = 80
        self.use = Usage()       # this session's total, across every model it routed to
        self.turn_use = Usage()  # the completed foreground turn only. Persisted with history
        self._usage_seen = {}    # backend cumulative counters already folded into `use`
        self.note = 'not started'
        self._backends, self._skills, self._reg, self._tools = {}, None, None, None
        self._subtools = None    # built only when sub-agents run on a smaller model than the turn
        self._subrec = None      # those tools recorded, for when sub-agents may write
        self._plain = []         # the unwrapped tools, which is what the briefing is written from
        self.poll_every, self._polled, self._poll_thread = float(poll_every or 0), 0.0, None
        self._monitor_thread = None
        # the folders something *else* is changing. Reviews run on the sub-agent model, read-only
        self.monitors = Monitors(host, get_backend=lambda: self._be_or_none('subagent'), get_tools=self._sub_plain)
        self.lock = threading.Lock()

`__init__` reaches these, directly or through each other, so they are patched before anything builds
an agent. Their own demo cells follow the harness below, which is what builds the agent they run
against. The rest of the class is in the order a reader wants rather than the order Python needs.

In [ ]:
#| export
@patch(as_prop=True)
def history_path(self:Agent):
    return None if self.cfg is None else self.cfg/f'{self.history_name}-history.jsonl'

In [ ]:
#| export
@patch
def spec_or_none(self:Agent, job='turn'):
    "`job`'s model spec, or `None` when its name does not resolve to one."
    try: return self.routing.spec(job)
    except Exception: return None

In [ ]:
#| export
@patch(as_prop=True)
def budget(self:Agent):
    "What the turn model can afford to be told. See `core.budget_for`."
    spec = self.spec_or_none()        # an unresolved name costs no tools and no channel
    return budget_for(spec, self.tool_max_len, tool_channel(spec))

In [ ]:
#| export
@patch(as_prop=True)
def registry(self:Agent):
    "The extension registry. The object outlives a reload, which re-reads only the extension files."
    if self._reg is None: self._reg = Registry(host=self.host, agent=self)
    if self.extensions and not self._reg.loaded:
        try: load(self._reg, self.host.roots, self.cfg, self.project_extensions, self.ext_paths)
        except Exception as e: self._reg.notes.append(f'extension loading failed: {agent_err(e)}')
    return self._reg

In [ ]:
#| export
@patch(as_prop=True)
def skills(self:Agent):
    "Every discovered skill, found once. Includes anything an extension registered."
    if self._skills is None:
        try:
            self._skills = discover(self.host.roots, self.cfg, extra=self.registry.skills)
        except Exception as e:
            self._skills = []
            self.note = f'skills unavailable ({agent_err(e)})'
    return self._skills

In [ ]:
#| export
@patch
def _be_or_none(self:Agent, job='turn'):
    "The backend for `job` if it can start, else None. What a tool asks, since a tool cannot raise usefully."
    try:
        b = self._be(job)
        return b if b.start() is not None else None
    except Exception: return None

In [ ]:
#| export
@patch
def _cloud_backend_or_none(self:Agent, model):
    "A started remote backend for one delegated fan-out, without changing routing."
    try:
        spec = self.routing._resolve(str(model))
        if spec.local: return None
        key = (spec.backend, spec.model_id)
        if key not in self._backends: self._backends[key] = make_backend(spec)
        backend = self._backends[key]
        return backend if backend.start() is not None else None
    except Exception: return None

In [ ]:
#| export
@patch
def _record(self:Agent, f):
    "Wrap one tool so its call is logged and its damage is measurable."
    name = getattr(f, '__name__', '?')

    @functools.wraps(f)   # both backends build the tool schema from the real signature
    def wrapper(*a, **kw):
        args = _named(f, a, kw)
        self._tool_calls_turn += 1
        if self.max_tool_calls is not None and self._tool_calls_turn > self.max_tool_calls:
            return ('Tool-call budget exhausted for this turn. Stop calling tools and '
                    'summarise the evidence and unfinished work now.')
        self.calls.append((name, args))
        meta = self._action_meta(name, args)
        act = self.activity.start(name, args, summary=summarise(f, args), **meta)
        self.registry.fire('before_tool', self, name, args)
        if name in WRITE_TOOLS:   # first touch only: later edits are part of one change
            if (p := args.get('path')):
                if p not in self.before: self.before[p] = self.host.text_at(p) or ''
            elif name == 'run_shell': self.snapshot_tree()
        nested = name in DELEGATE_TOOLS   # every call its sub-agent makes hangs off this one
        if nested: self._delegating.append(act.id)
        shelled = name == 'run_shell' and self._walked
        try: out = f(*a, **kw)
        except NotImplementedError as e:   # a raise ends the turn. A readable failure does not
            self.activity.finish(act, agent_err(e), ok=False)
            return err(f'{name} is not available here', e)
        except Exception as e:
            self.activity.finish(act, agent_err(e), ok=False)
            raise
        finally:
            if nested: self._delegating.pop()
            if shelled: self.settle_tree()
        self.activity.finish(act, out, ok=not failed(out))   # one spelling of failure, in one place
        self.registry.fire('after_tool', self, name, out)
        return out
    return wrapper

In [ ]:
#| export
@patch(as_prop=True)
def plan_path(self:Agent):
    "Per-session plan file beside the history log."
    if self.cfg is None or not getattr(self, 'session_id', None): return None
    return self.cfg/f'{self.history_name}-plans'/f'{self.session_id}.json'

In [ ]:
#| export
@patch
def _save_plan(self:Agent):
    p = self.plan_path
    if p is None: return
    try:
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(json.dumps(self.plan.dict(), ensure_ascii=False, indent=2))
    except Exception: pass
    if self.on_plan:
        try: self.on_plan(self.plan)
        except Exception: pass

In [ ]:
#| export
@patch(as_prop=True)
def subagent_budget(self:Agent):
    "What the sub-agent model can afford. Usually not the turn model's."
    spec = self.spec_or_none('subagent')
    if spec is None: return self.budget
    return budget_for(spec, self.tool_max_len, tool_channel(spec))

In [ ]:
#| export
@patch
def _sub_plain(self:Agent):
    "The tool list a sub-agent gets, sized to the model sub-agents run on."
    b = self.subagent_budget
    if b == self.budget:
        built = self.tools
        return built if self.subagent_writes else self._plain
    if self._subtools is None:
        self._subtools = tools_for(self.host, lambda: self.skills, list(self.registry.tools),
                                   mx=b.tool_max, drop=b.drop, get_spec=self.spec_or_none,
                                   on_media=self._drew)
    if not self.subagent_writes: return self._subtools
    if self._subrec is None: self._subrec = [self._record(t) for t in self._subtools]
    return self._subrec

`background` is where a `delegate_async` run lives until someone collects it. One per agent, built
on first use, and closed with the session so nothing keeps working after the last turn.

In [ ]:
#| export
@patch(as_prop=True)
def background(self:Agent):
    "The register async delegations run in, built on first use."
    if getattr(self, '_background', None) is None: self._background = Background()
    return self._background

In [ ]:
#| export
@patch(as_prop=True)
def tools(self:Agent):
    "Every tool the turn model can afford, built once and recorded. Rebuilt by `reload`."
    if self._tools is None:
        extra = list(self.registry.tools)
        if self.subagents:
            extra += subagent_tools(lambda: self._be_or_none('subagent'), self._sub_plain,
                                    lambda: self.skills, self._cloud_backend_or_none,
                                    lambda: self.subagent_writes,
                                    lambda: self.approvals.gate if self.approvals is not None else None,
                                    background=self.background)
        extra += plan_tools(lambda: self.plan, save=self._save_plan)
        b = self.budget
        extra += monitor_tools(lambda: self.monitors, mx=b.tool_max)
        plain = tools_for(self.host, lambda: self.skills, extra, mx=b.tool_max, drop=b.drop,
                          get_spec=self.spec_or_none, on_media=self._drew)
        if self.readonly: plain = read_only(plain, self.readonly_calls, effects=False, block=NO_SUB)
        self._plain = plain
        self._tools = [self._record(t) for t in plain]
    return self._tools

In [ ]:
#| export
@patch
def system_prompt(self:Agent):
    if self._sp: return self._sp
    if self._tools is None: self.tools
    # a skill body is 3k tokens of a 12k budget, and `read_skill` still reaches it
    inline = self.inline_skills if self.budget.inline else ()
    extra = ''
    if self.plan:
        extra = ('## Current plan\n\n' + self.plan.md() +
                 '\n\nWork the active todo; mark it done when finished; after a stop, '
                 'resume from the active item rather than rewriting the plan.')
    return system_prompt(self.host, self.skills, inline, tools=self._plain, extra=extra)

`memory_context` is the seam an application fills. The completer and the briefing both want the
notes a person has pinned to a surface, and only the application knows where those live, so the
agent asks itself rather than reaching for an attribute it hopes its embedder set.

In [ ]:
#| export
@patch
def memory_context(self:Agent, surface, max_chars=6000):
    "Durable user notes for one model surface. Nothing here; an embedder with a vault overrides it."
    return ''

In [ ]:
#| hide
# Every agent answers, so a caller never has to know whether its embedder filled the seam in.
# Ramabana's own answer is nothing, and nothing is a valid answer.
test_eq(Agent.memory_context(None, 'completion'), '')

In [ ]:
#| export
@patch
def _be(self:Agent, job='turn'):
    "The backend for `job`, built on first use and shared by every job on the same model."
    spec = self.routing.spec(job)
    key = (spec.backend, spec.model_id)
    if key not in self._backends:
        is_turn = key == (lambda s: (s.backend, s.model_id))(self.routing.spec('turn'))
        kw = {'multimodal': self.local_multimodal} if spec.runtime == 'litert' else {}
        if is_turn:
            kw.update(sp=self.system_prompt(), tools=self.tools, tool_max_len=self.tool_max_len,
                      approve=(self.approvals.gate if self.approvals is not None else None))
        self._backends[key] = make_backend(spec, **kw)
    return self._backends[key]

In [ ]:
#| export
#: bytes of the log read back for the live context. Whichever of the two bounds bites first wins,
#: so a log under the window behaves exactly as it did before there was one
HISTORY_TAIL = 8_000_000
HISTORY_TURNS = 2000

@patch
def _load_history(self:Agent):
    "The tail of the log"
    p = self.history_path
    if p is None or not p.exists(): return
    try:
        start = max(0, p.stat().st_size - HISTORY_TAIL)
        with p.open('rb') as f:
            f.seek(start); raw = f.read()
        lines = raw.decode('utf-8', 'replace').splitlines()
        if start and lines: del lines[0]   # the seek landed inside a line, and half a turn is not one
        self.history = [json.loads(line) for line in lines if line.strip()][-HISTORY_TURNS:]
    except Exception: self.history = []

In [ ]:
#| export
@patch
def _load_plan(self:Agent):
    p = self.plan_path
    if p is None or not p.exists(): return
    try: self.plan = Plan.from_dict(json.loads(p.read_text()))
    except Exception: pass

In [ ]:
#| export
@patch
def _drew(self: Agent, paths):
    "Record pictures a tool wrote this turn, and hand the frontend the paths at once."
    paths = list(paths)
    self._drawn = drawn = getattr(self, '_drawn', [])
    drawn.extend(paths)                # in place: two sub-agents may be drawing on two threads
    if self.on_media:
        try: self.on_media(paths)
        except Exception: pass

@patch(as_prop=True)
def resp_media(self: Agent):
    "Pictures the model itself returned on the newest turn, as `{'mime','data'}` dicts."
    be = self._be('turn')
    m = next((m for m in reversed(list(be.hist if be else []))
              if isinstance(m, dict) and m.get('role') == 'assistant'), None)
    return list((m or {}).get('media') or [])

@patch(as_prop=True)
def last_media(self: Agent):
    "Returns images from the latest turn as `{'mime','data'}` dicts, regardless of source (model or tool). All images are bytes for frontends. Paths are used only if a frontend supports them, avoiding duplicate writes."
    out = self.resp_media
    for p in getattr(self, '_drawn', []):
        p = Path(p)
        try: out.append({'mime': mime_for(p), 'data': p.read_bytes()})
        except OSError: pass
    return out

In [ ]:
#| export
def _named(f, a, kw=None):
    "Every argument as a dict, positional ones matched to their parameter names."
    kw = dict(kw or {})
    if not a: return kw
    try:
        import inspect
        return {**dict(zip(list(inspect.signature(f).parameters), a)), **kw}
    except Exception: return {**{'args': a}, **kw}


def _append(prompt, text):
    "Add `text` to a message that may be a list of content parts rather than a string."
    if not isinstance(prompt, (list, tuple)): return prompt + text
    parts = list(prompt)
    for i in range(len(parts) - 1, -1, -1):
        if isinstance(parts[i], str):
            parts[i] += text
            return parts
    return [*parts, text]


def _skill_text(skills, name):
    s = find(skills, name)
    return f'no skill matching {name!r}' if s is None else s.text()


def _with_notices(prompt):
    "Append the aai-coding style prompt notices, when the prompt earns any. Text prompts only."
    if not isinstance(prompt, str): return prompt
    return prompt + notices_block(prompt)

In [ ]:
#| export
def _run_store(self):
    if not hasattr(self, '_runs'):
        self._runs, self._runs_lock, self._foreground = {}, threading.RLock(), ''
    return self._runs

@patch
def runs(self:Agent, active=False):
    "Return registered root and child runs. `active` drops the ones that have finished."
    def live(r): return not r.terminal or any(live(child) for child in r.children)
    def row(r):
        d = r.dict()
        # A finished parent with a live child stays, or the child it is still running is orphaned.
        if active: d['children'] = [row(c) for c in r.children if live(c)]
        return d
    with getattr(self, '_runs_lock', threading.RLock()):
        return [row(r) for r in _run_store(self).values() if not active or live(r)]

In [ ]:
#| export
@patch(as_prop=True)
def busy(self:Agent): return bool(self.runs(active=True))

`busy` is here rather than beside the rest of the run bookkeeping because half the class asks it
first: `set_model`, `resume_session` and `cancel` all refuse while a turn is in flight.

Everything below runs a real `Agent` on a real model, replaying answers recorded once into
`chatcache/`. `recorded(record=False)` is what makes that safe: a question nobody recorded raises
instead of reaching a model, so this page cannot quietly start calling one. To add a question,
record it deliberately:

```sh
RISHI_RECORD_CHAT=1 uv run nbdev-test --path nbs/03_agent.ipynb
```

The host is a `MemHost` on a fixed `/proj`, because the system prompt is part of a recording's key.
A temporary folder would put a different absolute path in the briefing every run, and every replay
would miss.

In [ ]:
#| hide
import contextlib, tempfile
from pathlib import Path
from ramabana.core import JOBS
from ramabana.testing import recorded

replay = contextlib.ExitStack()
replay.enter_context(recorded())        # `record=False` unless $RISHI_RECORD_CHAT says otherwise

FILES = {'/proj/pkg/sizes.py': 'def threshold(n):\n    "Half of n."\n    return n // 2\n',
         '/proj/pkg/use.py': 'from .sizes import threshold\n\ndef budget(): return threshold(8192)\n'}
MODEL = 'claude/claude-sonnet-4-5'

def demo_agent(files=None, jobs=JOBS, **kw):
    "A real agent on a fixed `/proj`, whose model answers come from the recordings."
    kw.setdefault('cfg', Path(tempfile.mkdtemp()))     # a real config dir: history and plans land in it
    a = Agent(host=MemHost(dict(files or FILES)), model=MODEL, extensions=False, **kw)
    # every job, not just the turn: an unrouted one defaults to a local model, and this page
    # must not load one
    for j in jobs: a.routing.set(MODEL, j)
    return a

agent = demo_agent()

In [ ]:
bg = agent.background
test_eq(bg, agent.background)                              # one register per agent, not one per read
test_eq(bg.status(), [])
assert bg.result('run_nope').startswith('no delegation named')

# a background delegation is not the turn's child, so a turn that started one still goes idle
test_eq(agent.busy, False)

The briefing: what the agent is, where it is, how to work, and what it may read. The open folders
are in it, because a path outside them is refused and the model has to know which those are.

In [ ]:
sp = agent.system_prompt()
len(sp), sp[:120]

In [ ]:
sp = agent.system_prompt()
assert '/proj' in sp                                       # the folders it is confined to
assert 'search_code' in sp                                 # and the tools it was given
assert agent.system_prompt() == sp                         # stable, so a recording's key is stable

The agent caches its model tool list until an explicit invalidation.

In [ ]:
sorted(t.__name__ for t in agent.tools)

In [ ]:
names = {t.__name__ for t in agent.tools}
assert {'search_code', 'view_file', 'create_file'} <= names, sorted(names)
assert agent.tools is agent.tools                          # cached, not rebuilt per turn

The skill registry discovers skills once and caches the result. Extension skills are included before the cache is set.

In [ ]:
[s.name for s in agent.skills][:6]

In [ ]:
assert all(hasattr(s, 'text') for s in agent.skills)
test_eq([s.name for s in agent.skills], sorted(s.name for s in agent.skills))
assert agent.skills is agent.skills                       # found once, then cached

Extensions receive this state object. Disabled extension loading leaves it empty.

In [ ]:
agent.registry.tools, agent.registry.commands, agent.registry.notes

In [ ]:
test_eq(agent.registry.tools, [])
test_eq(dict(agent.registry.commands), {})
test_eq(agent.registry.notes, [])
assert agent.registry is agent.registry                   # loaded once

Which tool groups this model can afford. A small context window buys fewer schemas, and the
groups dropped are named rather than counted.

In [ ]:
agent.budget

In [ ]:
test_eq(agent.budget.drop, ())                             # a big window, so nothing is withheld
test_eq(agent.budget.inline, True)
test_eq(agent.budget.tool_max, MAX_TOOL_CHARS)
test_eq(agent.budget.note, 'full briefing')

Where this session's turns are written, and where the plan sits beside them.

In [ ]:
agent.history_path.name, agent.plan_path.name

In [ ]:
assert str(agent.history_path).endswith('-history.jsonl'), agent.history_path
assert agent.session_id in str(agent.plan_path)
test_eq(agent.plan_path.suffix, '.json')
test_eq(demo_agent(cfg=None).history_path, None)           # no config dir, so nothing is persisted

In [ ]:
#| export
@patch
def chat_or_none(self:Agent, job='turn'):
    "`job`'s live chat if a backend has already been built, without building one to find out."
    try:
        spec = self.routing.spec(job)
        b = self._backends.get((spec.backend, spec.model_id))
        return None if b is None else b.chat
    except Exception: return None

In [ ]:
#| export
@patch
def _forget(self:Agent):
    "Drop what is rebuilt from disk. The registry keeps whatever this process registered on it."
    self._skills = self._tools = self._subtools = self._subrec = None
    if self._reg is not None: self._reg.drop_loaded()

@patch
def reload(self:Agent):
    "Re-discover skills, extensions and tools. What a `/reload` command calls after editing them."
    self._forget()
    for b in self._backends.values(): b.close()
    self._backends.clear()
    return self

`add_tool` is the way in after `tools` is built: a turn can earn a tool it could not have
had at the start. `reload` and `refresh` re-read the extension files and keep it.

In [ ]:
#| export
@patch
def refresh(self:Agent):
    "Re-discover skills, extensions and tools, and re-brief a running turn backend in place."
    self._forget()
    spec = self.routing.spec('turn')
    b = self._backends.get((spec.backend, spec.model_id))
    if b is not None: b.refresh(self.system_prompt(), self.tools)
    return self

@patch
def add_tool(self:Agent, f):
    "Give this agent one more tool, now. Mid-turn, the running backend is re-briefed with it."
    self.registry.tool(f)
    self.refresh()
    return f

Drops the cached tool list so the next turn rebuilds it. This is what `/python` calls after giving
the host a kernel.

In [ ]:
before = agent.tools
agent.refresh()
agent.tools is before

In [ ]:
before = agent.tools
agent.refresh()
assert agent.tools is not before                           # rebuilt
test_eq({t.__name__ for t in agent.tools}, {t.__name__ for t in before})

A tool registered while the session is running survives both, and reaches the model.

In [ ]:
def hour(city: str) -> str:
    "What time it is in `city`."
    return f'noon in {city}'

agent.add_tool(hour)
assert 'hour' in {t.__name__ for t in agent.tools}
agent.reload(); agent.refresh()
test_eq('hour' in {t.__name__ for t in agent.tools}, True)   # neither throws it away
test_eq([t.__name__ for t in agent.registry.tools], ['hour'])

In [ ]:
#| export
@patch(as_prop=True)
def _delegating(self:Agent):
    "The delegate calls whose sub-agents are running on this thread, innermost last."
    # Per thread because `delegate_many` fans out over a threadpool. It only fans out for *reading*
    # sub-agents, which are not recorded, but a stack wrong under concurrency is not worth the saving.
    if not hasattr(self._nested, 'stack'): self._nested.stack = []
    return self._nested.stack

In [ ]:
#| export
@patch
def _action_meta(self:Agent, name, args):
    "Frontend-independent identity metadata for a call. Applications may override."
    return {'turn_id': self.current_turn_id, 'branch_id': self.current_branch_id,
            'parent_action_id': self._delegating[-1] if self._delegating else ''}

In [ ]:
#| export
@patch
def snapshot_tree(self:Agent):
    "Read the open folders, so `settle_tree` can tell what the command about to run moved."
    if self._walked: return True
    try: paths = [str(p) for p in self.host.walk()]
    except Exception as e:
        self.host.note(f'cannot watch what commands change: {agent_err(e)}')
        return False
    tree, n = {}, 0
    for p in paths:
        if (text := self.host.text_at(p)) is None: continue
        n += len(text)
        if n > SHELL_SNAPSHOT:
            self.host.note(f'not watching what commands change: the open folders hold over '
                           f'{SHELL_SNAPSHOT // 1_000_000}MB of text')
            return False
        tree[p] = text
    self._tree, self._walked = tree, True
    return True

In [ ]:
#| export
@patch
def settle_tree(self:Agent):
    "Decide what the command moved the moment it finishes, not when the turn does."
    if not self._walked: return
    tree, self._tree, self._walked = self._tree, {}, False
    for p, was in tree.items():
        now = self.host.text_at(p)
        if now is not None and now != was: self.before.setdefault(p, was)
    try: paths = [str(p) for p in self.host.walk()]     # a command also makes files, which
    except Exception: paths = []                        # no earlier snapshot can hold
    for p in paths:
        if p in tree or p in self.before: continue
        if self.host.text_at(p): self.before.setdefault(p, '')

In [ ]:
#| export
@patch
def changes(self:Agent):
    "`{path: (before, after)}` for every file this turn's write tools actually moved."
    out = {}
    for p, was in self.before.items():
        now = self.host.text_at(p)
        if now is not None and now != was: out[p] = (was, now)
    return out

The write diff records changed paths. It compares snapshots taken before and after the tool call.

In [ ]:
agent.changes()

In [ ]:
test_eq(agent.changes(), {})                               # nothing has written yet
agent.host.write('/proj/new.py', 'x = 1\n')
test_eq(agent.changes(), {})                               # a host write is not an agent write

In [ ]:
#| export
@patch(as_prop=True)
def backend(self:Agent): return self._be('turn')

In [ ]:
#| export
@patch(as_prop=True)
def chat(self:Agent):
    "The live chat object, or None. Kept for the frontends, which use it to cancel."
    return self._be('turn').chat

In [ ]:
#| export
@patch(as_prop=True)
def ready(self:Agent):
    "Whether the turn model is up. Asked of the backend rather than looked up in the cache."
    return self._be('turn').ready

Whether a backend is built and answering. False until something asks for a turn: an agent that
built an engine at construction would cost a model load to open a window.

In [ ]:
fresh = demo_agent()
fresh.ready, fresh.note

In [ ]:
fresh = demo_agent()
test_eq(fresh.ready, False)
test_eq(fresh.note, 'not started')

In [ ]:
#| export
@patch(as_prop=True)
def model(self:Agent): return self.routing.spec('turn')

The spec the turn routes to. `spec_or_none` is the same answer for any job, without raising when
nothing is configured for it.

In [ ]:
agent.model, [agent.spec_or_none(j).name for j in JOBS]

In [ ]:
test_eq(agent.model.runtime, 'claude')
test_eq(agent.model.model_id, 'claude-sonnet-4-5')
test_eq(agent.spec_or_none('turn'), agent.model)
test_eq(agent.spec_or_none('nonexistent-job'), agent.model)   # an unrouted job takes the turn model
# `demo_agent` pins every job to the recorded model, so this page never loads a local one
test_eq({agent.spec_or_none(j).model_id for j in JOBS}, {'claude-sonnet-4-5'})

In [ ]:
#| export
@patch
def start(self:Agent):
    "Build the turn backend, once. Returns it, or None with `note` explaining why not."
    b = self._be('turn')
    if b.start() is None:
        self.note = b.note
        return None
    # from the backend that is running, not from the routing table
    self.note = f'{model_note(b.spec)} · {len(self.tools)} tools'
    return b

Building the backend is deferred to the first turn, so opening a window costs nothing. `start`
is what does it, and `ready` and `note` say how it went.

In [ ]:
turn = demo_agent()
turn.start()
turn.ready, turn.note

In [ ]:
turn = demo_agent()
test_eq(turn.ready, False)
turn.start()
test_eq(turn.ready, True)
assert turn.backend is not None
test_eq(turn.chat_or_none('turn') is not None, True)

In [ ]:
#| export
@patch
def retry(self:Agent):
    "Forget a previous failure. A model that has since downloaded or been keyed is picked up."
    b = self._be('turn')
    b.retry()
    return self.start()

In [ ]:
#| export
@patch
def set_model(self:Agent, name, job='turn'):
    "Point `job` at `name`. A turn-model change carries the live conversation with it."
    if self.busy: raise RuntimeError('cannot change model while the assistant is working')
    previous = self.routing.spec(job)
    old = (previous.backend, previous.model_id)
    history = self._backends[old].snapshot_hist() if job == 'turn' and old in self._backends else []
    before = self.budget
    spec = self.routing.set(name, job)
    new = (spec.backend, spec.model_id)
    # tools and briefing are built from the turn model, not from its budget alone
    if job == 'turn' and (self.budget != before or new != old): self._tools = None
    if job == 'subagent': self._subtools = self._subrec = None
    if job == 'turn' and new != old:
        self._be('turn').resume_hist(history)
    still_used = {(self.routing.spec(j).backend, self.routing.spec(j).model_id) for j in JOBS}
    if old not in still_used and old in self._backends:
        self._backends.pop(old).close()
    self.note = f'{job} → {model_note(spec)}'
    return spec

In [ ]:
#| export
@patch
def set_subagent_writes(self:Agent, enabled):
    "Grant or withdraw sub-agent write access for this session."
    enabled = bool(enabled)   # refused mid-turn: a running delegation holds its tool list already
    if enabled == self.subagent_writes: return enabled
    if self.busy: raise RuntimeError('cannot change sub-agent writes while the assistant is working')
    self.subagent_writes = enabled
    self.note = f'sub-agent writes {"on" if enabled else "off"}'
    return enabled

In [ ]:
#| export
@patch
def set_local_multimodal(self:Agent, enabled):
    "Choose whether newly loaded LiteRT engines include media encoders."
    enabled = bool(enabled)
    if enabled == self.local_multimodal: return enabled
    if self.busy: raise RuntimeError('cannot change local multimodal while the assistant is working')
    self.local_multimodal = enabled
    for key in [k for k in self._backends if k[0] == 'litert']:
        self._backends.pop(key).close()
    self.note = f'local multimodal {"on" if enabled else "off"}'
    return enabled

In [ ]:
#| export
@patch
def lend_model(self:Agent):
    "Lend this session's engine to a host that builds its own chats, as a factory."
    if getattr(self.host, 'mk_chat', 'none of its business') is not None: return False

    def mk(model=None, **kw):
        from rishi import Chat
        spec = self._spec_for(model)   # honour the name: a local-only ask must not go to the cloud
        if spec is None: return Chat(model, **kw)
        b = self._backends.get((spec.backend, spec.model_id))
        engine = getattr(getattr(b, 'chat', None), 'engine', None)
        shared = {'engine': engine} if spec.local and engine is not None else {}
        return Chat(spec.model_id, runtime=spec.runtime, ctx_limit=spec.ctx, **shared, **kw)
    self.host.mk_chat = mk
    return True

In [ ]:
#| export
@patch
def _spec_for(self:Agent, model=None):
    "The `ModelSpec` a lent factory should build on. `None` means the cheap jobs' model."
    if model:
        try: return self.routing._resolve(str(model))
        except Exception: return None   # never substituted for a name asked for by name
    b = self._be_or_none('oneshot')
    return None if b is None or b.chat is None else b.spec

In [ ]:
#| export
@patch
def poll_watches(self:Agent, force=False):
    "Fire whatever the host has due, in a daemon thread, at most every `poll_every` seconds."
    import time
    if not self.poll_every and not force: return None
    if self._poll_thread is not None and self._poll_thread.is_alive(): return self._poll_thread
    now = time.monotonic()
    if not force and self._polled and now - self._polled < self.poll_every: return None
    self._polled = now

    def run():
        try: r = self.host.poll() or {}
        except NotImplementedError: return           # no watches here. Nothing to say about it
        except Exception as e: return self.host.note(f'could not poll watches: {agent_err(e)}')
        if r.get('ran'): self.host.note(f"{r['ran']} of {r.get('checked', 0)} watches fired; see memory_search")
    from fastcore.parallel import startthread
    self._poll_thread = startthread(run, daemon=True)
    self._poll_thread.name = 'ramabana-poll'
    return self._poll_thread

In [ ]:
#| export
@patch
def poll_monitors(self:Agent):
    "Look at every watched folder in a daemon thread. What it reviews reaches the turn after this one."
    if not self.monitors.all(): return None
    if self._monitor_thread is not None and self._monitor_thread.is_alive(): return self._monitor_thread

    def run():
        try: self.monitors.check()
        except Exception as e:
            try: self.host.note(f'could not check the watched folders: {agent_err(e)}')
            except Exception: pass
    from fastcore.parallel import startthread
    self._monitor_thread = startthread(run, daemon=True)
    self._monitor_thread.name = 'ramabana-monitor'
    return self._monitor_thread

A beat runs when no session does, so what it found is waiting rather than arriving. `beat_drain`
reads it once per session, under the session id, and a machine with no beat reads nothing.

In [ ]:
#| export
@patch(as_prop=True)
def beat(self:Agent):
    "The beat's database, opened once. None where no beat has ever run on this machine."
    if getattr(self, '_beat', 'unset') == 'unset':
        # `pob_path` is the one source of truth, so the beat and a session cannot open different files
        p = pob_path()
        self._beat = pob(p) if p.exists() else None
        # the reader is fixed when the beat is opened: `resume_session` renames the session, and a
        # reader that moved with it would replay notes this session already carried
        self._beat_reader = f'{POB_READER}:{self.session_id}'
    return self._beat

@patch
def beat_drain(self:Agent):
    "Notes the beat left that this session has not read. Empty when no beat runs here."
    if self.beat is None: return []
    return beat_notes(self.beat, reader=self._beat_reader)

In [ ]:
# no beat on this machine, so a turn carries nothing and nothing is opened
test_eq(agent.beat, None)
test_eq(agent.beat_drain(), [])
test_eq(beat_notice([]), '')
assert '<beat>' in beat_notice(['watches: 2 of 5 fired'])

In [ ]:
#| export
@patch
def _begin_turn(self:Agent, run=None):
    """This turn's own identity, before anything can go wrong with it.

    A turn stopped before `_prepare` still gets a row, and a row carrying the *previous* turn's id
    would collide with it in `conversation_parts`, where two rows sharing an id share group names.
    """
    rid = getattr(run, 'id', '')
    if rid and getattr(self, '_begun', None) == rid: return self.current_turn_id   # once per run
    self._begun = rid
    self.turn_use = Usage()                  # a stopped turn cannot inherit the last one's cost
    self._tool_calls_turn = 0                # applies even when a native engine owns the loop
    self.turn_seq += 1
    self.current_turn_id = f'{self.session_id}:turn_{self.turn_seq:06d}'
    self.activity.mark(self.current_turn_id)  # and so does `turn_md()`
    return self.current_turn_id

@patch
def _prepare(self:Agent, prompt):
    "Everything that happens before a message goes out: notices, hooks, and prospective compaction."
    self.before.clear()                    # `changes()` reports this turn, not the session
    self._drawn = []                       # pictures this turn's tools wrote, for the frontend
    self._walked, self._tree = False, {}
    self._begin_turn(current_run())
    self.checkpoints[self.current_turn_id] = {'before': self._be('turn').snapshot_hist(),
                                              'branch_id': self.current_branch_id}
    for old in list(self.checkpoints)[:-MAX_CHECKPOINTS]: self.checkpoints.pop(old, None)
    self.registry.fire('before_turn', self, prompt)
    self.poll_watches()
    reviews = self.monitors.drain()   # what a watched folder produced since the last turn
    self.poll_monitors()              # and the next look, whose reviews the next turn carries
    outgoing = _with_notices(prompt) if self.instruction_style == 'aai' else prompt
    request = request_text(prompt)
    requested, loaded = prompt_directives(request, self.tools, self.skills)
    route, plan = tool_plan(request)
    if requested:
        route = 'explicit'
        names = ', '.join(dict.fromkeys(name for name, _ in requested))
        plan = f'The user explicitly selected these tools: {names}. Use them before completing the task.'
    self._turn_plan = {'route': route, 'text': plan,
                       'tools': [name for name, _ in requested],
                       'skills': [skill.name for skill in loaded]}
    # planning has teeth: safe query tools run before generation. The model gets evidence
    preflights = []
    first = {'repo': 'search_code', 'web': 'web_search'}.get(route)
    if first: preflights.append((first, request))
    eager = {'search_code', 'web_search', 'research', 'memory_search', 'list_files'}
    preflights += [(name, query or request) for name, query in requested if name in eager]
    by_name = {getattr(t, '__name__', ''): t for t in self.tools}
    outgoing = _append(outgoing, f'\n\n<tool-plan route="{route}">{plan}</tool-plan>')
    if tool_channel(self.spec_or_none(), self.chat_or_none()) == 'tags': outgoing = _append(outgoing, OUTPUT_CONTRACT)
    for name, query in dict.fromkeys(preflights):
        tool = by_name.get(name)
        if tool is None: continue
        try: evidence = tool(query)
        except Exception as e: evidence = f'{name} failed: {agent_err(e)}'
        outgoing = _append(outgoing, f'\n\n<preflight-tool name="{name}">\n{evidence}\n</preflight-tool>')
    if reviews: outgoing = _append(outgoing, review_notice(reviews))
    # what the beat found while no session was running. Read once, under this session's id
    if (left := self.beat_drain()): outgoing = _append(outgoing, beat_notice(left))
    for skill in loaded:
        outgoing = _append(outgoing, f'\n\n<requested-skill name="{skill.name}">\n{skill.text()}\n</requested-skill>')
    b = self._be('turn')
    # measure the pending message too: a pasted notebook can cross the limit in one turn
    if self.compactor.auto and (self.compactor.due(b) or not b.fits(outgoing)): self.compact()
    # only when the turn cannot fit. Compaction cannot shrink the pending message
    if not b.fits(outgoing): outgoing = compact_notebook_context(outgoing, b.fits)
    if not b.fits(outgoing):
        projected = b.projected_tokens(outgoing)
        raise ValueError(f'input is too large for {b.spec.name}: about {projected:,} tokens '
                         f'with a {b.spec.ctx:,}-token context window')
    return outgoing

In [ ]:
#| export
@patch
def sessions(self:Agent):
    "Persisted conversations, newest first, with enough detail for a picker."
    grouped = {}
    for turn in self.history:
        sid = turn.get('session') or ''
        if not sid: continue
        grouped.setdefault(sid, []).append(turn)
    return [{'id': sid, 'turns': len(turns), 'at': turns[-1].get('at', 0),
             'model': turns[-1].get('model', ''),
             'title': str(turns[0].get('prompt', '')).replace('\n', ' ')[:72]}
            for sid, turns in sorted(grouped.items(), key=lambda x: x[1][-1].get('at', 0), reverse=True)]

@patch
def session_turns(self:Agent, sid):
    "One conversation's turns. From the turns in memory; a log-backed agent seeks to them instead."
    return [t for t in self.history if (t.get('session') or '') == str(sid)]

Every session on disk, newest first. `resume_session` takes one of these back up.

In [ ]:
[s['id'] for s in agent.sessions()][:3]

In [ ]:
test_eq(agent.sessions(), [])                              # a fresh config dir has no history yet
assert all('id' in s for s in agent.sessions())

In [ ]:
#| export
@patch
def session_added_roots(self:Agent, session_id):
    "Returns folders opened with `add_root` in order, read from the log for accurate session reconstruction. Does not reopen; see `resume_session`."
    out = []
    for turn in self.session_turns(session_id):
        # a turn that is not replayed does not widen the boundary either: honouring a root from a
        # turn whose context is left out would open a folder this session never agreed to
        if turn.get('state', 'complete') not in REPLAYED: continue
        for row in (turn.get('activity') or []):
            if row.get('tool') != 'add_root' or not row.get('ok', True): continue
            p = (row.get('args') or {}).get('path')
            if p and p not in out: out.append(p)
    return out

In [ ]:
#| export
@patch
def resume_session(self:Agent, selector='latest'):
    "Resume a persisted conversation by full/prefix id, or the newest with `latest`."
    if self.busy: raise RuntimeError('cannot resume while the assistant is working')
    choices = self.sessions()
    if not choices: raise KeyError('no saved sessions; start the CLI with --cfg or use its default config')
    selector = (selector or 'latest').strip()
    if selector == 'latest': picked = choices[0]
    else:
        matches = [s for s in choices if s['id'] == selector or s['id'].startswith(selector)]
        if len(matches) != 1: raise KeyError(f'session {selector!r} matched {len(matches)} conversations')
        picked = matches[0]
    self.resumed_roots = self.session_added_roots(picked['id'])
    turns = [t for t in self.session_turns(picked['id']) if t.get('state', 'complete') in REPLAYED]
    canonical = []
    for turn in turns:
        canonical.append({'role': 'user', 'content': str(turn.get('prompt', ''))})
        body = _resumed_acts(turn.get('activity')) + str(turn.get('reply') or '')
        if body.strip(): canonical.append({'role': 'assistant', 'content': body})
    if picked['model']: self.set_model(picked['model'])
    self._be('turn').resume_hist(canonical)
    self.session_id = picked['id']
    self.plan = Plan()
    self._load_plan()
    bit = f" · plan {self.plan.line()}" if self.plan else ''
    self.note = f"resumed {picked['id']} · {picked['turns']} turns · {picked['model']}{bit}"
    return picked

In [ ]:
#| export
REPLAYED = ('complete', 'failed')

@patch
def _remember(self:Agent, prompt, text, error='', state='complete'):
    turn = {'at': time.time(), 'session': getattr(self, 'session_id', '') or '', 'state': state,
            'turn_id': self.current_turn_id, 'branch_id': self.current_branch_id,
            'model': self._be('turn').spec.name,
            'prompt': str(prompt), 'reply': text, 'error': error,
            'plan': dict(getattr(self, '_turn_plan', {})),
            'usage': self.turn_use.dict(), 'usage_label': repr(self.turn_use),
            'activity': self.activity.rows(mark=self.activity._mark)}
    self.history.append(turn)
    del self.history[:-HISTORY_TURNS]
    if (p := self.history_path) is not None:
        try:
            p.parent.mkdir(parents=True, exist_ok=True)
            line = json.dumps(turn, ensure_ascii=False) + '\n'
            with p.open('a') as f:
                f.write(line)
                f.flush()
                end = f.tell()
            self._last_span = (max(0, end - len(line.encode())), end)
        except Exception: pass

In [ ]:
#| export
@patch
def _finish(self:Agent, text, prompt=''):
    b = self._be('turn')
    turn_use = Usage(model=b.use.model)
    backends = list(self._backends.items())
    if all(backend is not b for _, backend in backends): backends.append(((b.spec.backend, b.spec.model_id), b))
    for key, backend in backends:
        previous = self._usage_seen.get(key, Usage(model=backend.use.model))
        turn_use = turn_use + (backend.use - previous)
        self._usage_seen[key] = Usage(**backend.use.dict())
    turn_use.model = b.use.model or b.spec.model_id   # the foreground model is the label
    self.turn_use = turn_use
    self.use = self.use + turn_use
    self.registry.fire('after_turn', self, text)
    if self.current_turn_id in self.checkpoints: self.checkpoints[self.current_turn_id]['after'] = b.snapshot_hist()
    self._remember(prompt, text)
    return text

In [ ]:
#| export
@patch
def ask(self:Agent, prompt, **kw):
    "One turn. Returns the assistant's text, or the reason there isn't any."
    if self.start() is None: return self.note
    with self.lock:
        try:
            outgoing = self._prepare(prompt)
            return self._finish(self._be('turn').send(outgoing, **kw), prompt)
        except Exception as e:
            self.note = f'the assistant failed ({agent_err(e)})'
            self._remember(prompt, self.note, agent_err(e))
            return self.note

One turn. The prompt is composed, routed, sent, and what came back is remembered. The answer is a
recording, so nothing here reaches a model.

A recording holds what the call cost as well as what it said, and `CachedChat` folds it into the
same `use` counter a live chat keeps, so a replayed turn reports its own tokens. The read-outs
built on that are testable here. Compaction is not: a recorded chat has no `_recreate_conv`, so
`replace_hist` refuses it and `Compactor` returns the summary with the history untouched.

In [ ]:
asked = demo_agent()
answer = asked.ask('In one word, what colour is a clear sky?')
answer

In [ ]:
assert isinstance(answer, str) and answer.strip(), repr(answer)
assert 'lue' in answer, answer                             # blue, however it was capitalised
test_eq(len(asked.history), 1)                             # and the turn was remembered
test_eq(asked.history[-1]['prompt'], 'In one word, what colour is a clear sky?')
test_eq(asked.turn_use.model, 'claude-sonnet-4-5')
assert asked.turn_use.total > 0, asked.turn_use            # what the recorded call actually cost
assert asked.turn_use.input > asked.turn_use.output        # a briefing is longer than one word

In [ ]:
#| export
@patch
def stream(self:Agent, prompt, **kw):
    "One turn as an iterator of markdown chunks, for a frontend that can render as it arrives."
    if self.start() is None:
        yield self.note
        return
    with self.lock:
        try:
            outgoing = self._prepare(prompt)
            out = []
            for chunk in self._be('turn').stream(outgoing, **kw):
                out.append(chunk)
                yield chunk
            self._finish(''.join(out), prompt)
        except Exception as e:
            self.note = f'the assistant failed ({agent_err(e)})'
            self._remember(prompt, self.note, agent_err(e))
            yield f'\n\n{self.note}'

The same turn as an iterator, for a frontend that renders as the answer arrives.

This is the one turn on this page that cannot be replayed. A streamed reply is a generator, and
`CachedChat` refuses to store one rather than recording its `repr` as the answer. Under `recorded()`
the refusal comes back as the turn's text, which is what the cell below asserts. What `stream` does
with a live model is tested in `tests/`.

In [ ]:
streamed = demo_agent()
chunks = list(streamed.stream('In one word, what colour is a clear sky?'))
''.join(chunks)[:120]

In [ ]:
whole = ''.join(chunks)
assert chunks and whole, chunks                            # it yields something, always
# a generator cannot be stored. Recording, the refusal is the answer; replaying, there is no
# recording to find. Either way it comes back as text rather than raising into the turn
assert ('lue' in whole or 'cannot record a generator' in whole or 'no recording' in whole), whole
test_eq(len(streamed.history), 1)                          # a streamed turn is still a turn

In [ ]:
#| export
@patch
def compose(self:Agent, prompt, context='', screen='', image=None, context_path=''):
    "One message from what the frontend can supply: the notebook, the screen as text, the screen as a picture."
    parts = []
    # a text-only LiteRT engine cannot accept image parts. Leave a truthful marker
    local_text_only = bool(image) and self.model.runtime == 'litert' and not self.local_multimodal
    if local_text_only: parts.append('[Image attachment omitted: local multimodal is disabled.]')
    if context:
        # the path is operational context: without it a small model invents one
        attr = f' path="{context_path}"' if context_path else ''
        parts.append(f'<notebook{attr}>\n{context}\n</notebook>')
    if screen: parts.append(f'<screen>\n{screen}\n</screen>')
    parts.append(f'<user-request>\n{prompt}\n</user-request>')
    ask = '\n\n'.join(parts)
    if not image or local_text_only: return ask
    media = list(image) if isinstance(image, (list, tuple)) else [image]
    return [*media, ask]

The frontend composes a message around the question: an open notebook, a screenshot, a file the
user pointed at. `compose` is where those become one prompt.

In [ ]:
shaped = demo_agent()
shaped.compose('what does this do?', context='def a(): return 1', context_path='/proj/a.py')

In [ ]:
one = shaped.compose('what does this do?', context='def a(): return 1', context_path='/proj/a.py')
assert '<user-request>\nwhat does this do?\n</user-request>' in one
assert '<notebook path="/proj/a.py">' in one and 'def a(): return 1' in one
# the request is always tagged, so the model can tell the question from what came with it
test_eq(shaped.compose('bare'), '<user-request>\nbare\n</user-request>')

In [ ]:
#| export
@patch
def ask_with(self:Agent, prompt, context='', screen='', image=None, context_path='', **kw):
    "One turn with the frontend's context attached. Blocking. See `stream_with` for the live one."
    return self.ask(self.compose(prompt, context, screen, image, context_path), **kw)

In [ ]:
#| export
@patch
def stream_with(self:Agent, prompt, context='', screen='', image=None, context_path='', **kw):
    "The same turn, as an iterator of markdown chunks."
    return self.stream(self.compose(prompt, context, screen, image, context_path), **kw)

In [ ]:
#| export
@patch
def cancel(self:Agent):
    "Stops an active turn and returns if one was running, regardless of backend abort capability."
    running = self.busy
    if self.approvals is not None: self.approvals.cancel_all('the turn was stopped')
    self._be('turn').cancel()
    return running

In [ ]:
#| export
@patch
def close(self:Agent):
    for b in list(self._backends.values()):
        try: b.close()
        except Exception: pass
    self._backends.clear()

In [ ]:
#| export
@patch
def context_parts(self:Agent, turn_id, stage='after'):
    "Each message in a checkpoint is a branchable part, grouped to keep calls and results together. Part IDs persist across reloads."
    cp = self.checkpoints.get(str(turn_id))
    if cp is None or stage not in cp: raise KeyError(f'no {stage} checkpoint for {turn_id}')
    out, group = [], 0
    for i, m in enumerate(cp[stage]):
        role = m.get('role') if isinstance(m, dict) else ''
        kind = ('calls' if role == 'assistant' and m.get('tool_calls') else
                'result' if role == 'tool' else role or 'other')
        if kind != 'result': group += 1        # a result belongs to the call above it
        body = m.get('content') if isinstance(m, dict) else ''
        out.append({'part_id': f'{turn_id}:{i}', 'turn_id': str(turn_id), 'stage': stage,
                    'index': i, 'kind': kind, 'group': f'{turn_id}:g{group}',
                    'preview': (body if isinstance(body, str) else '')[:200]})
    return out

The same idea against a live checkpoint rather than the log: one part per message, with a call and
its results grouped so they move together. A checkpoint that was never taken is a `KeyError` rather
than an empty list.

In [ ]:
checked = demo_agent()
for stage in ('after', 'before'):
    try: checked.context_parts('no-such-turn', stage)
    except KeyError as e: assert 'checkpoint' in str(e), (stage, e)
    else: assert False, f'{stage} should have refused'

In [ ]:
checked = demo_agent()
for stage in ('after', 'before'):
    try: checked.context_parts('no-such-turn', stage)
    except KeyError as e: assert 'checkpoint' in str(e), (stage, e)
    else: assert False, f'{stage} should have refused'

In [ ]:
#| export
@patch
def conversation_parts(self:Agent, sid=None):
    "A stored conversation as ordered, editable parts reconstructed from the turn log, with consistent part IDs across processes."
    sid = sid or self.session_id
    out = []
    for turn in [t for t in self.session_turns(sid) if t.get('state', 'complete') in REPLAYED]:
        tid = str(turn.get('turn_id') or '')
        group = 0
        def part(kind, text, editable, extra=None):
            nonlocal group
            out.append({'part_id': f'{tid}:{len(out)}', 'turn_id': tid, 'kind': kind,
                        'group': f'{tid}:g{group}', 'editable': editable,
                        'text': text, **(extra or {})})
        part('user', str(turn.get('prompt') or ''), True)
        for act in (turn.get('activity') or []):
            group += 1
            part('call', str(act.get('line') or act.get('summary') or act.get('tool') or ''), False,
                 {'tool': act.get('tool') or '', 'action_id': act.get('action_id') or act.get('id') or '',
                  'parent_action_id': act.get('parent_action_id') or '', 'ok': bool(act.get('ok'))})
            part('result', str(act.get('detail') or ''), False, {'tool': act.get('tool') or ''})
        group += 1
        if turn.get('reply'): part('assistant', str(turn['reply']), True)
    return out

One stored conversation as the parts a person can keep, drop or rewrite. Built from the turn log
rather than a live checkpoint, because the conversation someone wants to reshape is usually one they
have just resumed, and a resumed assistant holds no snapshots.

In [ ]:
shaper = demo_agent()
shaper.ask('In one word, what colour is a clear sky?')
[(p['part_id'].split(':')[-1], p['kind'], p['editable']) for p in shaper.conversation_parts()]

In [ ]:
shaper = demo_agent()
shaper.ask('In one word, what colour is a clear sky?')
parts = shaper.conversation_parts()
test_eq([p['kind'] for p in parts], ['user', 'assistant'])
assert all(p['part_id'] and p['turn_id'] and p['group'] for p in parts)
assert all(p['editable'] for p in parts)                   # prose may be rewritten; a call may not
test_eq([p['part_id'] for p in shaper.conversation_parts()], [p['part_id'] for p in parts])

In [ ]:
#| export
@patch
def compile_conversation(self:Agent, sid=None, manifest=None, rewrites=None):
    """Provider messages for a reshaped conversation: the stored turns, minus what a person
    discarded, with their own words in place of any prose they rewrote. A call and its result
    move together, and neither can be rewritten -- editing them would claim work that never ran.
    """
    parts, manifest = self.conversation_parts(sid), dict(manifest or {})
    rewrites = {str(k): str(v) for k, v in (rewrites or {}).items()}
    bad = [p for p in manifest.values() if p not in BRANCH_POLICIES]
    if bad: raise ValueError(f'unknown context policy {bad[0]!r}')
    fixed = {p['part_id'] for p in parts if not p['editable']}
    refused = sorted(set(rewrites) & fixed)
    if refused: raise ValueError(f'{refused[0]} is not prose, so it cannot be rewritten')
    unknown = sorted(set(rewrites) - {p['part_id'] for p in parts})
    if unknown: raise ValueError(f'no part {unknown[0]} in this conversation')
    forced = {p['group'] for p in parts if manifest.get(p['part_id']) == 'keep'}
    gone = {p['group'] for p in parts if manifest.get(p['part_id']) == 'discard'} - forced
    kept = [p for p in parts if p['group'] not in gone]
    msgs = []
    for p in kept:
        text = rewrites.get(p['part_id'], p['text'])
        if p['kind'] == 'user': msgs.append({'role': 'user', 'content': text})
        elif p['kind'] == 'assistant': msgs.append({'role': 'assistant', 'content': text})
        elif p['kind'] == 'call':
            msgs.append({'role': 'assistant', 'content': '',
                         'tool_calls': [{'id': p['part_id'], 'type': 'function',
                                         'function': {'name': p.get('tool') or 'tool', 'arguments': '{}'}}]})
        else: msgs.append({'role': 'tool', 'tool_call_id': msgs[-1]['tool_calls'][0]['id'] if msgs and msgs[-1].get('tool_calls') else p['part_id'],
                           'content': text})
    return {'messages': msgs, 'parts': parts, 'kept': len(kept),
            'omitted': len(parts) - len(kept), 'rewritten': sorted(rewrites),
            'groups': sorted({p['group'] for p in parts}),
            'adjusted': sorted({p['part_id'] for p in parts
                                if p['group'] in gone and manifest.get(p['part_id']) != 'discard'})}

In [ ]:
#| export
@patch
def reshape(self:Agent, sid=None, manifest=None, rewrites=None, branch_id='', revision=None):
    "Make a reshaped conversation the active branch, without touching what was recorded."
    compiled = self.compile_conversation(sid, manifest, rewrites)
    branch_id = branch_id or f'branch_{uuid.uuid4().hex[:8]}'
    self._branch_hist.setdefault(self.current_branch_id, self._be('turn').snapshot_hist())
    self._be('turn').resume_hist(compiled['messages'])
    parent, self.current_branch_id = self.current_branch_id, branch_id
    self._branch_hist[branch_id] = compiled['messages']
    record = self.save_branch(branch_id, revision=revision, parent_branch_id=parent,
                             parent_turn_id=str(sid or self.session_id), stage='reshaped',
                             manifest=dict(manifest or {}), rewrites=dict(rewrites or {}))
    return {**record, 'omitted': compiled['omitted'], 'kept': compiled['kept'],
            'rewritten': compiled['rewritten'], 'adjusted': compiled['adjusted']}

In [ ]:
#| export
@patch
def compile_context(self:Agent, turn_id, stage='after', part_id='', manifest=None):
    "A provider message is a checkpoint truncated at `part_id`, with each part's policy applied. Calls and results form a decision unit, not a partial exchange."
    cp = self.checkpoints.get(str(turn_id))
    if cp is None or stage not in cp: raise KeyError(f'no {stage} checkpoint for {turn_id}')
    msgs, manifest = cp[stage], dict(manifest or {})
    bad = [p for p in manifest.values() if p not in BRANCH_POLICIES]
    if bad: raise ValueError(f'unknown context policy {bad[0]!r}')
    parts = self.context_parts(turn_id, stage)
    if part_id:
        cut = next((p for p in parts if p['part_id'] == str(part_id)), None)
        if cut is None: raise ValueError(f'no part {part_id} in {turn_id} {stage}')
        last = max(p['index'] for p in parts if p['group'] == cut['group'])
        parts = [p for p in parts if p['index'] <= last]
    forced = {p['group'] for p in parts if manifest.get(p['part_id']) == 'keep'}
    gone = {p['group'] for p in parts if manifest.get(p['part_id']) == 'discard'} - forced
    kept = [p for p in parts if p['group'] not in gone]
    return {'messages': [msgs[p['index']] for p in kept], 'parts': parts, 'omitted': len(parts) - len(kept), 'kept': len(kept),
            'groups': sorted({p['group'] for p in parts}), 
            'adjusted': sorted({p['part_id'] for p in parts if p['group'] in gone and manifest.get(p['part_id']) != 'discard'})}

In [ ]:
#| export
@patch
def fork(self:Agent, turn_id, stage='after', branch_id='', part_id='', manifest=None, revision=None):
    "Fork model context from a captured turn boundary, or a part inside it, and make it active."
    compiled = self.compile_context(turn_id, stage, part_id, manifest)
    branch_id = branch_id or f'branch_{uuid.uuid4().hex[:8]}'
    self._branch_hist.setdefault(self.current_branch_id, self._be('turn').snapshot_hist())
    self._be('turn').restore_hist(compiled['messages'])
    parent, self.current_branch_id = self.current_branch_id, branch_id
    self._branch_hist[branch_id] = compiled['messages']
    record = self.save_branch(branch_id, revision=revision, parent_branch_id=parent, parent_turn_id=str(turn_id),
                parent_part_id=str(part_id or ''), stage=stage, manifest=dict(manifest or {}))
    return {**record, 'omitted': compiled['omitted'], 'kept': compiled['kept'], 'adjusted': compiled['adjusted'], 
            'parent_turn_id': str(turn_id), 'stage': stage}

In [ ]:
#| export
@patch
def switch_branch(self:Agent, branch_id):
    """Make another branch active by rebuilding its context, never by copying it. A branch is
    its parent point plus its manifest, so recompiling is what switching means."""
    branch_id = str(branch_id)
    if branch_id == self.current_branch_id: return self.branch_meta(branch_id)
    held = self._branch_hist.get(branch_id)
    if held is None:
        meta = self.branch_meta(branch_id)
        if not meta['parent_turn_id']: raise KeyError(f'branch {branch_id} has no recorded parent point')
        held = self.compile_context(meta['parent_turn_id'], meta['stage'] or 'after',
                                    meta['parent_part_id'], meta['manifest'])['messages']
    self._branch_hist.setdefault(self.current_branch_id, self._be('turn').snapshot_hist())
    self._be('turn').restore_hist(held)
    self._branch_hist[branch_id] = held
    self.current_branch_id = branch_id
    return self.branch_meta(branch_id)

In [ ]:
#| export
@patch
def undo_turn(self:Agent, turn_id, branch_id=''):
    """A turn undone is a branch that stops before it. The turn stays in canonical history --
    undo is not deletion, and redo is switching back rather than replaying."""
    return self.fork(turn_id, 'before', branch_id)

In [ ]:
#| export
@patch
def revise(self:Agent, turn_id, text, branch_id=''):
    "Fork after a turn and replace its prose response with a user-authored revision."
    branch = self.fork(turn_id, 'after', branch_id)
    self._be('turn').revise_last_assistant(text)
    self._branch_hist[branch['branch_id']] = self._be('turn').snapshot_hist()
    branch['revision_text'] = str(text)
    return branch

In [ ]:
#| export
@patch
def oneshot(self:Agent, prompt, sp='', job='oneshot', max_tokens=None):
    "A question on whichever model `job` routes to, in a conversation that is thrown away."
    b = self._be_or_none(job)
    # the job, not the transport's method name: a summary that failed said `one-shot failed`
    return '' if b is None else b.oneshot(prompt, sp, max_tokens, job=job)

A question that needs no history and no tools. It routes to the cheap job rather than the turn
model, so a classification never costs a context window.

In [ ]:
asker = demo_agent()
said = asker.oneshot('Reply with exactly the word: yes')
said

In [ ]:
assert 'yes' in said.lower(), said
test_eq(len(asker.history), 0)                             # a oneshot is not a turn, so nothing is logged
# a job routed to a model that cannot answer returns nothing rather than raising into the turn
class _Unavailable:
    def _be_or_none(self, job): return None
test_eq(Agent.oneshot(_Unavailable(), 'anything at all'), '')

In [ ]:
#| export
@patch
def classify(self:Agent, text, labels):
    "One label for `text`, on the cheap model. Returns the matched label, or the raw reply."
    out = self.oneshot(f'{text}\n\nChoose exactly one label from: {", ".join(labels)}.',
                       'Reply with only the single best label and nothing else.', 'classify', 32).lower()
    return next((l for l in labels if l.lower() in out), out.strip())

In [ ]:
#| export
@patch
def summarise(self:Agent, text, sp='Summarise concisely. Output only the summary.'):
    return self.oneshot(text, sp, 'summary')

The summarizer runs in a separate agent and context window. Its recording key includes its own conversation history.

In [ ]:
summariser = demo_agent()
shorter = summariser.summarise('The user asked about thresholds. The agent read pkg/sizes.py and answered 4096.')
shorter

In [ ]:
assert isinstance(shorter, str) and shorter.strip(), repr(shorter)
assert len(shorter) < 600, len(shorter)                    # shorter than what it was given

In [ ]:
#| export
@patch
def compact(self:Agent, extra=''):
    "Compact the conversation now. Returns the summary text, or `''` with `compactor.note` set."
    b = self._be('turn')
    if b.chat is None:
        self.compactor.note = 'nothing to compact: the model is not running'
        return ''
    sub = self._be_or_none('summary')
    summary_backend = sub if sub is not None else b
    summary_output = min(1024, max(256, summary_backend.spec.ctx // 4))
    summariser = summary_backend.oneshot
    text = self.compactor.compact(
        b, lambda p, sp: summariser(p, sp, summary_output), extra,
        summary_ctx=summary_backend.spec.ctx, summary_output=summary_output,
        summary_count=summary_backend.count_tokens)
    if text: self.registry.fire('compact', self, text)
    self.note = self.compactor.note
    return text

In [ ]:
#| export
@patch(as_prop=True)
def pct_full(self:Agent):
    try: return self._be('turn').pct_full
    except Exception: return 0.0

In [ ]:
#| export
@patch(as_prop=True)
def context_used(self:Agent):
    try: return self._be('turn').used_tokens
    except Exception: return 0

In [ ]:
#| export
@patch
def turn_md(self:Agent, title='what I did'):
    "This turn's tool calls as foldable markdown, to save alongside the answer in a cell."
    return self.activity.md(mark=self.activity._mark, title=title)

The activity feed as markdown, folded. What a frontend that can show a tool result draws.

In [ ]:
print(asked.turn_md())

In [ ]:
assert isinstance(asked.turn_md(), str)
test_eq(asked.turn_md(title='nothing happened'), '')       # an empty turn draws nothing

In [ ]:
#| export
@patch
def turn_lines(self:Agent):
    "This turn's tool calls as plain summary lines, for a pane that cannot fold."
    return self.activity.lines(mark=self.activity._mark)

The activity feed for the turn, one line per call. `turn_md` is the same thing folded for a
frontend that can show a result.

In [ ]:
agent.turn_lines(), agent.calls

In [ ]:
test_eq(agent.turn_lines(), [])
test_eq(agent.calls, [])
assert isinstance(agent.turn_md(), str)

In [ ]:
#| export
@patch(as_prop=True)
def problems(self:Agent):
    "Everything that went wrong and had nowhere to be reported, newest last."
    out = []
    for b in self._backends.values():
        for p in b.problems:
            if p not in out: out.append(p)
    if (n := self.compactor.note).startswith('compaction') and n not in out: out.append(n)
    return out[-10:]

Gathered from every backend rather than kept on the agent, so a model that failed to load says so
wherever it was routed to.

In [ ]:
agent.problems, agent.pct_full, agent.context_used

In [ ]:
test_eq(agent.problems, [])
test_eq(agent.pct_full, 0.0)                               # nothing asked yet, so nothing is used
test_eq(agent.context_used, 0)

# what the window held, replayed from the recording. Not a literal: the briefing grows whenever a
# tool is added, and the recording is made again when it does
assert asked.context_used > 1000, asked.context_used
assert 0 < asked.pct_full < 1, asked.pct_full              # a fraction of the window, not a percent

# occupancy and what the turn was billed are different numbers, and this backend reports both. The
# bill counts cache reads and sums every step; the window holds what the session last measured
assert asked.context_used != asked.turn_use.total, 'the two counters collapsed into one'
assert abs(asked.context_used - asked.turn_use.total) < asked.turn_use.total // 4

In [ ]:
#| export
@patch
def clear_problems(self:Agent):
    "Forget them, for a frontend that has shown them."
    for b in self._backends.values(): b.problems.clear()
    return self

Drops whatever the backends are complaining about, so a fixed problem stops being reported.

In [ ]:
agent.clear_problems()
agent.problems

In [ ]:
test_eq(agent.problems, [])

In [ ]:
#| export
@patch
def status(self:Agent):
    "Everything a status bar or an `/agent` command wants, in one dict."
    return {'ready': self.ready, 'busy': self.busy, 'note': self.note,
            'problems': self.problems,
            'model': self.model.name, 'model_note': model_note(self.model),
            'budget': self.budget.note, 'tool_budget': self.tool_budget,
            'approve': getattr(self.approvals, 'mode', ''), 'step_budget': self.step_budget,
            'tool_calls': self._tool_calls_turn, 'tool_limit': self.max_tool_calls, 'step_limit': self.max_steps,   # a tool withheld for a small window is invisible otherwise
            'ntools': len(self.tools), 'nskills': len(self.skills),
            'pct_full': round(self.pct_full, 3), 'compactions': self.compactor.count,
            'use': self.use.dict(), 'usage': repr(self.use),
            'plan': self.plan.dict(), 'plan_line': self.plan.line(),
            'activity': self.activity.rows(40),
            'approval': (self.approvals.pending.dict() if self.approvals is not None
                         and self.approvals.pending is not None else None),
            'calls': [{'tool': t, 'args': str(a)[:300]} for t, a in self.calls[-40:]]}

In [ ]:
#| export
@patch
def command(self:Agent, line):
    "Run a slash command. Returns text to show, or None when the command is unknown."
    line = (line or '').strip().lstrip('/')
    name, _, arg = line.partition(' ')
    arg = arg.strip()
    if name == 'model':
        if not arg: return self.routing.summary()
        job, _, m = arg.partition(' ')
        try: return f'{model_note(self.set_model(m or job, job if m else "turn"))}'
        except Exception as e: return agent_err(e)
    if name == 'sessions':
        rows = self.sessions()
        if rows: return '\n'.join(f"{s['id']}  {s['turns']:>3} turns  {s['model']:<20} {s['title']}" for s in rows)
        where = str(self.history_path) if self.history_path is not None else '(history disabled: no cfg directory)'
        return f'no saved sessions in {where}; a session is saved after its first completed turn'
    if name == 'resume':
        try:
            s = self.resume_session(arg or 'latest')
            return f"resumed {s['id']} · {s['turns']} turns · {s['model']}"
        except Exception as e: return agent_err(e)
    if name == 'models':
        rows = available_models(include_legacy=arg.lower() in ('all', 'legacy'))
        if not rows: return 'no models available'
        width = max(len(r['value']) for r in rows)
        return '\n'.join(f"{'*' if r['value'] == self.model.name else ' '} {r['value']:<{width}}  {r['provider']:<12} {r['source']}" for r in rows)
    if name == 'subagents':
        if arg:
            want = arg.strip().lower()
            if want not in ('on', 'off', 'read', 'write'): return "say /subagents on|off"
            self.subagent_writes = want in ('on', 'write')
        return ('sub-agents may write, run commands and run Python, behind this session\'s approvals'
                if self.subagent_writes else
                'sub-agents are read-only: they report what they found and change nothing')
    if name == 'cost': return repr(self.use)
    if name == 'compact':
        t = self.compact(arg)
        return f'{self.compactor.note}\n\n{t}' if t else self.compactor.note
    if name == 'skills': return '\n'.join(f'{s.name:16} {s.source:8} {s.description[:90]}' for s in self.skills) or 'no skills found'
    if name == 'skill': return clip(_skill_text(self.skills, arg))
    if name == 'tools': return '\n'.join(sorted(getattr(t, '__name__', '?') for t in self.tools))
    if name == 'extensions': return '\n'.join(self.registry.notes) or 'no extensions loaded'
    if name == 'reload':
        self.reload()
        return f'reloaded: {len(self.tools)} tools, {len(self.skills)} skills'
    if name in ('plan', 'todos'):
        if not arg: return self.plan.md()
        if arg.lower() in ('clear', 'reset', 'none'): self.plan.clear(); self._save_plan(); return 'plan cleared'
        if '|' in arg:
            title, _, rest = arg.partition('|')
            items = [x.strip() for x in rest.split('|') if x.strip()]
        else:
            lines = [ln.strip() for ln in arg.splitlines() if ln.strip()]
            title, items = (lines[0], lines[1:]) if lines else (arg, [])
        self.plan.set(title, items); self._save_plan()
        return self.plan.md()
    if name == 'todo':
        if not arg: return self.plan.md()
        head, _, rest = arg.partition(' ')
        rest = rest.strip()
        if self.plan.find(head) is not None and rest:
            status, _, note = rest.partition(' ')
            if status in TODO_STATUSES:
                try: self.plan.update(head, status=status, note=note.strip() or None)
                except Exception as e: return agent_err(e)
                self._save_plan(); return self.plan.md()
        try: self.plan.add(arg)
        except Exception as e: return agent_err(e)
        self._save_plan(); return self.plan.md()
    if name in self.registry.commands:
        fn, _ = self.registry.commands[name]
        try: return fn(self, arg)
        except Exception as e: return agent_err(e)
    return None

In [ ]:
#| export
_agent_status, _agent_command = Agent.status, Agent.command

#: Seconds a cancelled run is given to stop before terminating. A class attribute, so it is set
#: on any `Agent` before a run exists and survives a caller raising it.
Agent.cancel_grace = .25

def _stream_chunk(out, chunk):
    text = ''.join(out)
    if chunk == text: return ''
    return chunk[len(text):] if chunk.startswith(text) else chunk

@patch
def run(self:Agent, run_id=''):
    "Find a registered run, or the foreground root when `run_id` is empty."
    roots = list(_run_store(self).values())
    if not run_id: run_id = self._foreground
    def walk(r):
        if r.id == run_id:return r
        return next((hit for child in r.children if (hit := walk(child)) is not None), None)
    return next((hit for root in roots if (hit := walk(root)) is not None), None)

@patch
def _new_run(self:Agent, prompt):
    _run_store(self)
    with self._runs_lock:
        current = self.run()
        if current is not None and not current.terminal: raise RuntimeError('the assistant is already running')
        r = Run(f'run_{uuid.uuid4().hex[:12]}', question=str(prompt), model=self.model.name, grace=self.cancel_grace)
        self._runs[r.id], self._foreground = r, r.id
        for old in list(self._runs)[:-100]: self._runs.pop(old, None)
        return r

@patch
def _release_run(self:Agent, run):
    if run.state not in ('detached', 'terminated'): return
    for key, backend in list(self._backends.items()):
        if backend is run.backend: self._backends.pop(key, None)

@patch
def ask(self:Agent, prompt, **kw):
    "One registered turn. A stopped turn is recorded, and not replayed."
    run, kept = self._new_run(prompt), [False]
    def keep(state, text='', error=''):
        "Write the row once, whichever way the turn ended. The flag is set by a write that happened."
        if kept[0]: return
        self._remember(prompt, text, error, state=state)
        kept[0] = True
    try:
        self._begin_turn(run)
        if self.start() is None:
            run.finish('failed'); keep('failed', self.note, self.note); return self.note
        backend = self._be('turn')
        if not run.start(backend): keep('cancelled'); return run.dict()
        with run_context(run):
            outgoing = self._prepare(prompt)
            text = backend.send(outgoing, run=run, **kw)
        if run.cancelled:
            run.finish(); keep('cancelled'); return run.dict()
        run.finish()
        out = self._finish(text, prompt)
        kept[0] = True
        return out
    except Exception as e:
        if run.cancelled:
            run.finish(); keep('cancelled'); return run.dict()
        run.finish('failed')
        self.note = f'the assistant failed ({agent_err(e)})'
        keep('failed', self.note, agent_err(e))
        return self.note
    finally:
        if not run.terminal: run.finish('cancelled')
        keep('abandoned')

@patch
def stream(self:Agent, prompt, on_registered=None, **kw):
    "One registered turn as markdown chunks. A stopped turn is recorded, and not replayed."
    run, out, kept = self._new_run(prompt), [], [False]
    def keep(state, text='', error=''):
        "Write the row once, whichever way the turn ended. The flag is set by a write that happened."
        if kept[0]: return
        self._remember(prompt, text, error, state=state)
        kept[0] = True
    try:
        self._begin_turn(run)                  
        if self.start() is None:
            run.finish('failed')
            keep('failed', self.note, self.note)
            yield self.note; return
        backend = self._be('turn')
        if not run.start(backend): keep('cancelled'); return
        if on_registered is not None: on_registered(run)
        if run.cancelled: run.finish(); keep('cancelled'); return
        with run_context(run):
            outgoing = self._prepare(prompt)
            for chunk in backend.stream(outgoing, run=run, **kw):
                if run.cancelled: break
                chunk = _stream_chunk(out, chunk)
                if chunk: out.append(chunk); yield chunk
        run.finish()
        if run.cancelled: keep('cancelled', ''.join(out))
        else:
            self._finish(''.join(out), prompt)   
            kept[0] = True
    except Exception as e:
        if run.cancelled: run.finish(); keep('cancelled', ''.join(out)); return
        run.finish('failed')
        self.note = f'the assistant failed ({agent_err(e)})'
        keep('failed', self.note, agent_err(e))
        yield f'\n\n{self.note}'
    finally:
        if not run.terminal: run.finish('cancelled')
        keep('abandoned', ''.join(out))

@patch
def cancel(self:Agent, run_id='', grace=None):
    "Cancel one run, or the foreground root, and return its structured terminal state."
    run = self.run(run_id)
    if run is None:return {'id': run_id or None, 'state': 'idle', 'children': []}
    if self.approvals is not None:self.approvals.cancel_all('the run was cancelled')
    run.cancel(self.cancel_grace if grace is None else grace)
    self._release_run(run)
    return run.dict()

@patch
def terminate(self:Agent, run_id=''):
    "Request resource termination for one run without waiting on the provider."
    run = self.run(run_id)
    if run is None:return {'id': run_id or None, 'state': 'idle', 'children': []}
    run.terminate(); self._release_run(run)
    return run.dict()

@patch
def status(self:Agent):
    out = _agent_status(self)
    out['runs'] = self.runs(active=True)
    out['busy'] = bool(out['runs'])
    return out

@patch
def command(self:Agent, line):
    raw = (line or '').strip().lstrip('/')
    name, _, arg = raw.partition(' ')
    if name == 'stop': return json.dumps(self.cancel(arg.strip()), ensure_ascii=False)
    if name == 'runs': return json.dumps(self.runs(active=arg.strip() != 'all'), ensure_ascii=False)
    return _agent_command(self, line)


A turn that was stopped is still a turn that happened. The row carries `state`, so the log keeps it
and a resume leaves it out: a half-streamed reply must not go back as a whole one. `REPLAYED` is
which states return as context, and a row written before the field existed reads as `complete`.

In [ ]:
import json

def _rows(a):
    "What survives a restart, which is the file rather than `a.history`."
    p = a.history_path
    return [json.loads(l) for l in p.read_text().splitlines()] if p and p.exists() else []

dropped = demo_agent()
g = dropped.stream('a question nobody waited for')
chunk = next(g)
g.close()                                    # the caller drops the generator: `GeneratorExit`

rows = _rows(dropped)
test_eq(len(rows), 1)
test_eq(rows[0]['state'], 'abandoned')
test_eq(rows[0]['reply'], chunk)             # what it had already streamed is kept
test_eq(dropped.busy, False)                 # ...and the run it left behind is finished

In [ ]:
# only a turn that ran to the end, or failed, goes back to the model on a resume. Not the
# constant: what a resume actually puts back, for each state a row can carry
kept = demo_agent()
list(kept.stream('a turn that finished'))
row = kept.history[-1]

def replayed(state):
    kept.history = [dict(row, state=state, prompt=f'a {state} turn')]
    kept.resume_session(kept.session_id)
    b = kept._be('turn')
    return ' '.join(m.get('content', '') for m in (b._resume_hist or b.hist or []))

for state in REPLAYED: assert f'a {state} turn' in replayed(state), state
for state in ('cancelled', 'abandoned'):
    assert f'a {state} turn' not in replayed(state), state

# a row from before `state` existed has none, and must keep reading as a complete turn
kept.history = [{k: v for k, v in row.items() if k != 'state'}]
kept.resume_session(kept.session_id)
b = kept._be('turn')
assert 'a turn that finished' in ' '.join(m.get('content', '') for m in (b._resume_hist or b.hist or []))

Points a job at another model. Changing the turn model carries the live conversation across, and
rebuilds the tools when the new window can afford a different set.

In [ ]:
routed = demo_agent()
routed.set_model('gpt-4.1', 'subagent')
routed.spec_or_none('subagent')

In [ ]:
routed = demo_agent()
before = routed.model
routed.set_model('gpt-4.1', 'subagent')
test_eq(routed.spec_or_none('subagent').model_id, 'openai/gpt-4.1')
test_eq(routed.model, before)                              # the turn model is untouched
test_fail(lambda: routed.set_model('no-such-model'), contains='unknown model')

Whether a delegated task may write. Off by default: a sub-agent cannot see the conversation that
sent it, so anything it changes is a change nobody reviewed.

In [ ]:
writer = demo_agent()
writer.subagent_writes

In [ ]:
test_eq(writer.subagent_writes, False)                     # off unless a session turns it on
writer.set_subagent_writes(True)
test_eq(writer.subagent_writes, True)
writer.set_subagent_writes(False)
test_eq(writer.subagent_writes, False)

Compaction is one call on the summary model, and it says what it did rather than silently dropping
half the conversation.

In [ ]:
compacted = demo_agent()
compacted.ask('In one word, what colour is a clear sky?')
compacted.compact(), compacted.compactor.note

In [ ]:
compacted = demo_agent()
compacted.ask('In one word, what colour is a clear sky?')
before = len(compacted.history)
got = compacted.compact()
assert isinstance(got, (bool, str, dict)), type(got)
assert isinstance(compacted.compactor.note, str)
test_eq(len(compacted.history), before)                    # compaction moves the window, not the log

Session resume restores the persisted conversation. It does not restore roots added after the session started.

In [ ]:
resumed = demo_agent(cfg=asked.cfg)
[s['id'] for s in resumed.sessions()]

In [ ]:
resumed = demo_agent(cfg=asked.cfg)
assert [s['id'] for s in resumed.sessions()], 'the earlier turn should have been persisted'
sid = resumed.sessions()[0]['id']
resumed.resume_session(sid)
test_eq(resumed.session_id, sid)
assert resumed.history, 'the turns came back with it'
test_eq(resumed.session_added_roots(sid), [])

Compaction is one call on the summary model, and it reports what it did either way.

In [ ]:
#| export
@patch
def commands(self:Agent):
    "Every command name, built-in and registered, for a help line or an autocomplete."
    return sorted({'model', 'models', 'sessions', 'resume', 'cost', 'compact', 'skills', 'skill', 'tools', 'extensions', 'reload',
                   'subagents', 'plan', 'todos', 'todo', *self.registry.commands})

### Pictures the model sent back, and pictures its tools drew

A generating model returns images on the response, not in its text. `rishi` carries them as `media` on the assistant's history message. `resp_media` is where a frontend collects them after the turn. Streaming yields markdown chunks. It does not yield these images.

A tool writes a file and returns its path because a tool result is text. `on_media` fires when the tool writes the file. A frontend can draw the picture while the turn is running. `last_media` combines both routes as bytes for a frontend that cannot draw a path.

`fake_agent` from [testing](04_testing.ipynb) is a real `Agent` whose every job routes to a scripted backend. A turn can be driven here with no model at all.

In [ ]:
agent, be = fake_agent(host, replies=['`threshold` is in `ramabana/runtime.py`.'])
agent.ask('where is the compaction threshold?')

'`threshold` is in `ramabana/runtime.py`.'

In [ ]:
# a turn that generated nothing has nothing to collect
agent, be = fake_agent(host, replies=['no pictures here'])
agent.ask('hello')
test_eq(agent.last_media, [])

be.chat.hist.append({'role': 'assistant', 'content': 'here you go', 'media': [{'mime': 'image/png', 'data': b'\x89PNG\r\n\x1a\nx'}]})
test_eq([m['mime'] for m in agent.last_media], ['image/png'])
first, _ = fake_agent(host); second, _ = fake_agent(host)
assert first.session_id != second.session_id

In [ ]:
# a tool's picture reaches the frontend the moment it is written, and only by path:
# `resp_media` never carries it, so a frontend that drew it live does not save it again
import tempfile
with tempfile.TemporaryDirectory() as d:
    shot = Path(d)/'shot.png'
    shot.write_bytes(b'\x89PNG\r\n\x1a\nx')
    drawer, _ = fake_agent(host, replies=['drawn'])
    seen = []
    drawer.on_media = seen.append
    drawer._drew([shot])
    test_eq(seen, [[shot]])
    test_eq(drawer.resp_media, [])
    test_eq([m['mime'] for m in drawer.last_media], ['image/png'])

The turn was routed, and the route is recorded rather than inferred: this prompt asked about the repository. The plan said search first.

In [ ]:
agent._turn_plan

Planning has teeth. A repo question runs `search_code` *before* generation and hands the model the evidence, instead of advising it to choose a tool it may ignore. This is what the composed message shows:

In [ ]:
print(be.sent[0][-400:])

 question mark, so it is a question. Make only the tool calls needed to answer it, then answer it, then stop -- do not start the work it implies.
</system-reminder>

<tool-plan route="repo">Use search_code first. Read the matching source if needed. Do not web-search a question about Leela or the open repository.</tool-plan>

<preflight-tool name="search_code">
no matches (memory)
</preflight-tool>


Every call is recorded as it happens. The feed and the saved cell agree with each other.

In [ ]:
agent.calls, agent.turn_lines()

([('search_code', {'query': 'where is the compaction threshold?'})],
 ['🔍 Search where is the compaction threshold?'])

`status()` is everything a status bar or an `/agent` command wants, in one dict.

In [ ]:
{k: v for k, v in agent.status().items() if k in ('ready', 'busy', 'model', 'ntools', 'pct_full', 'usage')}

{'ready': True,
 'busy': False,
 'model': 'ornith-9b',
 'ntools': 14,
 'pct_full': 0.015,
 'usage': '15 tok · in 10 · out 5 · model'}

A write tool is snapshotted before it runs. `changes()` reports the file rather than the claim about the file. A tool that said it succeeded and changed nothing does not appear.

In [ ]:
agent.ask('create a file')
create = next(t for t in agent.tools if t.__name__ == 'create_file')
create('/proj/b.py', 'def b(): return 2\n')
agent.changes()

{'/proj/b.py': ('', 'def b(): return 2\n')}

A shell command is snapshotted before it runs and settled the moment it returns. The turn goes on thinking long after that, and the snapshot covers every open folder, so deciding at the end of the turn credited it with whatever anyone else wrote meanwhile. `changes()` is what the undo transaction and the host's journal are built from, so that was an offer to revert somebody else's work.

In [ ]:
shell_host = MemHost({'/proj/a.py': 'a = 1\n', '/proj/elsewhere.py': 'e = 1\n'})
shell_agent, _ = fake_agent(shell_host)
shell_agent.before.clear()
run_shell = next(t for t in shell_agent.tools if t.__name__ == 'run_shell')

def _ran(command, cwd=None, timeout=120):
    shell_host.files['/proj/a.py'] = 'a = 2\n'
    return 0, 'done'

shell_host.run_cmd = _ran
run_shell('touch a.py')
shell_host.files['/proj/elsewhere.py'] = 'e = 2\n'   # an editor, a rebase, an agent in another repository
test_eq(shell_agent.changes(), {'/proj/a.py': ('a = 1\n', 'a = 2\n')})

Slash commands live here rather than in the frontends, because both of them need every one of these and a command that exists in the terminal and not the browser is exactly the drift worth preventing. An unknown command returns `None`, which is how a frontend knows to pass it on.

In [ ]:
agent.command('/tools').splitlines()[:4]

['create_file', 'create_skill', 'delegate_parallel', 'delegate_search']

In [ ]:
test_eq(agent.command('/nope'), None)
print(agent.command('/model'))

turn        ornith-9b · local · 32k ctx
inline      ornith-9b · local · 32k ctx
completion  qwen-4b · local · 32k ctx
classify    qwen-4b · local · 32k ctx
summary     qwen-4b · local · 32k ctx
subagent    ornith-9b · local · 32k ctx


The cheap jobs route away from the turn model: a classification or a summary runs on whatever `classify` and `summary` point at, in a conversation that is thrown away.

In [ ]:
agent.oneshot('is this a question?'), agent.summarise('a long transcript')

('ONESHOT:is this a question?', 'ONESHOT:a long transcript')

In [ ]:
agent.compact(), agent.compactor.note

('ONESHOT:<conversation>\n<message index=1 role=use',
 'compacted 4 message(s), kept 4')

Problems are gathered from every backend rather than kept on the agent, because most of them happen on the *cheap* model. A compaction that could not run, a completion the engine refused. Where the caller cannot raise and used to get `''` as the whole story.

In [ ]:
agent.problems, agent.pct_full, agent.context_used

([], 0.015)

## Inline completion

Inline completion is the argument for routing in one feature: four lines of code, fired constantly, has to feel instant, worth approximately nothing per call. It runs on the local model in a throwaway conversation, and never automatically. The kernel's own completions are instant and correct, while this costs a forward pass.

In [ ]:
#| export
COMPLETE_SP = """You are a code completion engine inside an editor. You are given the code before \
the cursor in <before> and the code after it in <after>.

Reply with ONLY the code that belongs at the cursor. No explanation, no markdown fence, no \
repetition of <before> or <after>. Keep it short -- finish the current expression, statement or \
short block and stop. Match the surrounding indentation and style exactly. If nothing sensible \
belongs there, reply with nothing at all."""

MAX_COMPLETION_LINES = 4     # a suggestion longer than this is a guess about the design, not a completion
COMPLETION_TOKENS = 96
CTX_BEFORE, CTX_AFTER = 2000, 600   # chars of surrounding code sent as context


def _strip_echo(before, out):
    "Drop a re-emitted tail of `before` from the front of `out`. Models like to restate the line they continue."
    tail = before[-200:]
    for n in range(len(tail), 0, -1):
        if out.startswith(tail[-n:]): return out[n:]
    return out

In [ ]:
#| export
def _fence_tail(text):
    "What follows an *unterminated* fence, or None. Everything before the opener is prose."
    if '```' not in text: return None
    head, _, rest = text.rpartition('```')
    if '```' in head and head.count('```') % 2: return None   # a complete block: fenced_blocks has it
    return rest.partition('\n')[2] if '\n' in rest else ''


def _clean(text, before, max_lines):
    "A raw model reply as something safe to insert: fences off, prose off, echo off, `max_lines` long."
    from fastcore.xtras import fenced_blocks
    text = text or ''
    if (blocks := fenced_blocks(text)): out = blocks[-1][1]
    elif (tail := _fence_tail(text)) is not None: out = tail
    else: out = text
    out = _strip_echo(before, (out or '').strip('\n'))
    lines = out.split('\n')[:max_lines]
    while lines and not lines[-1].strip(): lines.pop()
    return '\n'.join(lines)

`_clean` is what makes a raw reply safe to insert. A completion model asked for bare code fences it anyway, restates the line it is continuing, and keeps going past the point where it is guessing about the design.

In [ ]:
before = 'def threshold(ctx, reserve=RESERVE):\n    '
_clean('```\n    if not ctx: return None\n    return max(1, ctx - reserve)\n```', before, 4)

'if not ctx: return None\n    return max(1, ctx - reserve)'

In [ ]:
test_eq(_clean('    ' + 'x = 1', '    ', 4), 'x = 1')       # the echoed indent is dropped
_clean('a\nb\nc\nd\ne\nf', '', 3)

'a\nb\nc'

The shape a small local model actually produces is prose, then a fence the token cap cut off before it closed. Everything before the opener is the essay, and inserting *that* into the buffer is the one outcome worse than suggesting nothing.

In [ ]:
reply = 'Looking at the code, `area(r)` returns nothing. Let me fix it:\n```python\n    return math.pi * r**2'
_clean(reply, before, 4)

'return math.pi * r**2'

In [ ]:
test_eq(_clean('I will explain first.\n```python', before, 4), '')   # nothing but prose: suggest nothing
_clean('```\ncomplete block\n```', '', 4)

'complete block'

In [ ]:
#| export
class Completer:
    "Inline completion: the local model, a throwaway conversation, and only when asked for."

    def __init__(self, agent, max_lines=MAX_COMPLETION_LINES, max_tokens=COMPLETION_TOKENS):
        self.a, self.max_lines, self.max_tokens = agent, max_lines, max_tokens
        self.note = 'not asked yet'

    @property
    def ready(self):
        b = self.a._be_or_none('completion')
        return b is not None

    def _prompt(self, code, pos, lang, context=''):
        try: variables = self.a.host.list_vars()
        except Exception: variables = ''
        support = ''
        if context: support += f'<related_code>\n{context[-6000:]}\n</related_code>\n'
        if variables: support += f'<runtime_variables>\n{variables[:4000]}\n</runtime_variables>\n'
        try: memory = self.a.memory_context('completion', max_chars=6000)
        except Exception: memory = ''   # a vault that cannot be read costs a note, never the completion
        if memory: support += f'<user_memory>\n{memory}\n</user_memory>\n'
        return (f'Language: {lang}\n\n{support}<before>\n{code[:pos][-CTX_BEFORE:]}\n</before>\n'
                f'<after>\n{code[pos:][:CTX_AFTER]}\n</after>')

    def complete(self, code, pos, lang='python', context=''):
        "The text to insert at `pos` in `code`, or `''` with `note` saying why there isn't any."
        b = self.a._be_or_none('completion')
        if b is None:
            self.note = 'no completion model available'
            return ''
        if b.busy:   # one engine, one generation: declining beats queueing behind a tool loop
            self.note = 'model busy -- it is mid-turn'
            return ''
        text = b.oneshot(self._prompt(code, pos, lang, context), COMPLETE_SP, self.max_tokens)
        if not text:
            self.note = b.note if not b.ready else 'no suggestion'
            return ''
        out = _clean(text, code[:pos], self.max_lines)
        self.note = f'{len(out.splitlines())} line(s) from {b.spec.name}' if out else 'no suggestion'
        return out

In [ ]:
#| hide
# The completer asks the agent, rather than reaching for an attribute only one embedder ever set:
# `self.a.ws` resolved nowhere in Ramabana, so the notes never reached a completion prompt anywhere.
class _Embedder:
    "What an application with a vault looks like from the completer's side."
    host = NullHost()
    def memory_context(self, surface, max_chars=6000): return f'note for {surface}'
assert '<user_memory>\nnote for completion\n</user_memory>' in Completer(_Embedder())._prompt('x = 1', 5, 'python')
class _Bare:
    host = NullHost()
    memory_context = Agent.memory_context
assert '<user_memory>' not in Completer(_Bare())._prompt('x = 1', 5, 'python')

The completer asks the `completion` backend, and says why there is nothing when there is nothing. An editor that silently inserts nothing is indistinguishable from a broken one.

In [ ]:
comp = Completer(agent)
comp.ready, comp.complete('def a():\n    ', 12)

(True, 'ONESHOT:Language: python\n\n<before>\ndef a():')

In [ ]:
comp.note

'4 line(s) from fake'

In [ ]:
#| export
_TOOL_MIN, _TOOL_MAX = 20, 400
_STEP_MIN, _STEP_MAX = 8, 80


def _limit(value, lo, hi, default):
    if str(value or '').lower() == 'auto': return 'auto'
    try: return max(lo, min(hi, int(value)))
    except (TypeError, ValueError): return default

_agent_init_limits = Agent.__init__
def _init_limits(self, *args, max_tool_calls='auto', max_steps='auto', **kwargs):
    _agent_init_limits(self, *args, **kwargs)
    self.tool_budget = _limit(max_tool_calls, _TOOL_MIN, _TOOL_MAX, 'auto')
    self.step_budget = _limit(max_steps, _STEP_MIN, _STEP_MAX, 'auto')
    self.max_tool_calls = self.max_steps = None
Agent.__init__ = _init_limits

_agent_prepare_limits = Agent._prepare
def _prepare_limits(self, prompt):
    self.max_tool_calls = None if self.tool_budget == 'auto' else self.tool_budget
    self.max_steps = None if self.step_budget == 'auto' else self.step_budget
    backend = self._be('turn')
    if hasattr(backend, 'max_steps'): backend.max_steps = self.max_steps
    if getattr(backend, 'chat', None) is not None and hasattr(backend.chat, 'max_steps'):
        backend.chat.max_steps = self.max_steps
    return _agent_prepare_limits(self, prompt)
Agent._prepare = _prepare_limits

_agent_command_limits = Agent.command
def _command_limits(self, line):
    raw = (line or '').strip().lstrip('/')
    name, _, arg = raw.partition(' ')
    arg = arg.strip()
    if name in ('tool-budget', 'steps'):
        attr, lo, hi = ('tool_budget', _TOOL_MIN, _TOOL_MAX) if name == 'tool-budget' else ('step_budget', _STEP_MIN, _STEP_MAX)
        if arg:
            value = _limit(arg, lo, hi, getattr(self, attr))
            if value == getattr(self, attr) and str(arg).lower() != 'auto' and not str(arg).isdigit(): return f'usage: /{name} [auto|{lo}..{hi}]'
            setattr(self, attr, value)
        value = getattr(self, attr)
        active = self.max_tool_calls if name == 'tool-budget' else self.max_steps
        return '{}: {} (last turn: {})'.format(name, value, active if active is not None else 'automatic')
    return _agent_command_limits(self, line)
Agent.command = _command_limits


_agent_commands_limits = Agent.commands
def _commands_limits(self): return sorted(set(_agent_commands_limits(self)) | {'tool-budget', 'steps'})
Agent.commands = _commands_limits

In [ ]:
#| hide
# `model=MODEL` only so no local engine is loaded: none of this makes a model call
auto = Agent(NullHost(), model=MODEL, extensions=False)
auto._prepare('test')
auto_call = auto._record(lambda: 'ok')
test_eq(auto.max_tool_calls, None)
test_eq([auto_call() for _ in range(81)][-1], 'ok')

limited = Agent(NullHost(), model=MODEL, extensions=False, max_tool_calls=20)
limited._prepare('test')
limited_call = limited._record(lambda: 'ok')
test_eq([limited_call() for _ in range(21)][-1].startswith('Tool-call budget exhausted'), True)

limited_steps = Agent(NullHost(), model=MODEL, extensions=False, max_steps=20)
backend = limited_steps._be('turn')
backend.max_steps = 40
backend.chat = type('Chat', (), {'max_steps': 40})()
limited_steps._prepare('test')
test_eq((backend.max_steps, backend.chat.max_steps), (20, 20))
limited_steps.step_budget = 'auto'
limited_steps._prepare('test')
test_eq((backend.max_steps, backend.chat.max_steps), (None, None))

`runs(active=True)` lists what is running. A subagent that has finished is not that, even while its parent still is.

In [ ]:
from ramabana.testing import fake_agent
from ramabana.runtime import Run

a, be = fake_agent(replies=['done'])
root = a._new_run('parent'); root.start(be)
done = Run('child_done', kind='child', parent=root)
going = Run('child_going', kind='child', parent=root)
done.start(be); done.finish(); going.start(be)
test_eq([c['id'] for c in a.runs(active=True)[0]['children']], ['child_going'])
test_eq(len(a.runs()[0]['children']), 2)
deep = Run('grandchild', kind='child', parent=done); deep.start(be)
kids = {c['id']: c for c in a.runs(active=True)[0]['children']}
test_eq(sorted(kids), ['child_done', 'child_going'])
test_eq([c['id'] for c in kids['child_done']['children']], ['grandchild'])

In [ ]:
#| export
_agent_record = Agent._record

@patch
def _record(self:Agent, f):
    wrapped = _agent_record(self, f)
    @functools.wraps(wrapped)
    def call(*a, **kw):
        run = current_run()
        if run is not None and run.cancelled:return 'Run cancelled; this tool call was not started.'
        return wrapped(*a, **kw)
    return call

Cancellation removes a root from active status. Partial output is not finished or remembered.

In [ ]:
from ramabana.testing import FakeBackend, fake_agent

class _BlockingBackend(FakeBackend):
    def __init__(self):
        super().__init__(replies=['unused'])
        self.first, self.release, self.cancelled = threading.Event(), threading.Event(), False
    def stream(self, msg, run=None, **kw):
        yield 'partial'; self.first.set(); self.release.wait(); yield 'late'
    def cancel(self): self.cancelled = True; self.release.set(); return True

a, _ = fake_agent(replies=['unused'])
be = _BlockingBackend()
a._be = lambda job='turn': be
a._be_or_none = lambda job='turn': be
a.cancel_grace = .05
finished, remembered, chunks = [], [], []
a._finish = lambda *args, **kw: finished.append(args)
a._remember = lambda *args, **kw: remembered.append((args, kw))
t = threading.Thread(target=lambda: chunks.extend(a.stream('stop me')), daemon=True)
t.start(); assert be.first.wait(.2)
run_id = a.status()['runs'][0]['id']
result = a.cancel(run_id); t.join(.2)
test_eq(result['state'], 'cancelled')
test_eq(chunks, ['partial'])
test_eq((finished, a.status()['runs']), ([], []))
test_eq(len(remembered), 1)
test_eq(remembered[0][0][:1], ('stop me',))          # the prompt is kept
test_eq(remembered[0][1], {'state': 'cancelled'})    # ...and marked, so a resume leaves it out

class _SnapshotBackend(FakeBackend):
    def _stream(self, msg, **kw): yield from ('The ', 'The answer', 'The answer')

a, _ = fake_agent(replies=['unused'])
be = _SnapshotBackend()
a._be = a._be_or_none = lambda job='turn': be
test_eq(list(a.stream('repeat')), ['The ', 'answer'])
test_eq(a.history[-1]['reply'], 'The answer')

a, _ = fake_agent(replies=['unused'])
be = _BlockingBackend()
a._be = a._be_or_none = lambda job='turn': be
registered, chunks = [], []
def registered_before_output(run):
    registered.append(run.id)
    run.cancel(0)
test_eq(list(a.stream('register first', on_registered=registered_before_output)), [])
test_eq((len(registered), be.first.is_set()), (1, False))

a, _ = fake_agent(replies=['unused'])
be = _BlockingBackend()
a._be = a._be_or_none = lambda job='turn': be
def registration_failed(run): raise RuntimeError('registration failed')
test_eq(list(a.stream('bad registration', on_registered=registration_failed)),
        ['\n\nthe assistant failed (RuntimeError: registration failed)'])
test_eq(a.runs(active=True), [])
test_eq(be.first.is_set(), False)


In [ ]:
#| export
_agent_commands_runs, _agent_close_runs = Agent.commands, Agent.close

@patch
def commands(self:Agent): return sorted(set(_agent_commands_runs(self)) | {'stop', 'runs'})

@patch
def close(self:Agent):
    "Cancel active runs before closing their backends."
    backends = list(self._backends.values())
    if getattr(self, '_background', None) is not None: self._background.close()
    if self.approvals is not None: self.approvals.close()
    for row in self.runs(active=True): self.cancel(row['id'])
    evicted = [b for b in backends if all(b is not live for live in self._backends.values())]
    _agent_close_runs(self)
    for backend in evicted:
        try: backend.close()
        except Exception: pass


Stops the turn in flight, and reports the run it stopped. Nothing is running here, so the report is
an idle root with no children.

In [ ]:
asked.cancel()

In [ ]:
# a later cell re-patches `cancel` to report the run tree, which is what a frontend draws
test_eq(asked.cancel(), {'id': None, 'state': 'idle', 'children': []})
test_eq(asked.busy, False)

Releases every backend the session built. An agent that is closed answers `ready` False again.

In [ ]:
asked.close()
asked.ready

In [ ]:
asked.close()
test_eq(asked.ready, False)
asked.close()                                              # closing twice is not an error

In [ ]:
a, be = fake_agent(replies=['done'])
test_eq({'stop', 'runs'} <= set(a.commands()), True)
order = []
be.cancel = lambda: order.append('cancel') or True
be.close = lambda: order.append('close')
a._backends[('fake', 'fake/model')] = be
run = a._new_run('shutdown'); run.start(be)
a.close()
test_eq(order[:2], ['cancel', 'close'])


In [ ]:
prompt = '''<notebook path=a.ipynb>
...
</notebook>

<user-request>
why is it reading a.ipynb and df_norm
</user-request>

<tool-plan route="direct">Answer directly.</tool-plan>
<system-reminder>Verify it.</system-reminder>'''
test_eq(request_text(prompt), 'why is it reading a.ipynb and df_norm')
test_eq(tool_plan(request_text(prompt))[0], 'direct')
test_eq(request_text('plain request'), 'plain request')
test_eq(request_text('<user-request></user-request>'), '')

In [ ]:
class _BusyAgent:
    runs = Agent.runs
    busy = Agent.busy

for state in ('pending', 'running', 'cancelling'):
    a = _BusyAgent()
    root = Run('root', state=state)
    a._runs, a._runs_lock, a._foreground = {'root': root}, threading.RLock(), 'root'
    assert a.busy

for state in ('completed', 'failed', 'cancelled', 'detached', 'terminated'):
    a = _BusyAgent()
    root = Run('root', state=state)
    a._runs, a._runs_lock, a._foreground = {'root': root}, threading.RLock(), 'root'
    assert not a.busy

a = _BusyAgent()
root = Run('root', state='completed')
root.child('delegated').state = 'running'
a._runs, a._runs_lock, a._foreground = {'root': root}, threading.RLock(), 'root'
assert a.busy
root.children[0].finish()
assert not a.busy

### The agent ledger

[panjika](https://github.com/vedicreader/panjika) is an append-only record of which agent session touched which file, and where that change went in git. Every harness working in a repository writes to the same one, so Ramabana's turns sit on one timeline with Claude Code's and Codex's.

`_remember` hands the turn over whole, outside the history lock, because panjika appends with one `write` syscall and asks git questions of its own. A turn is recorded only where a ledger already exists: `panjika install` makes one, and Ramabana never creates one uninvited. panjika is not a dependency, so a missing install is a skipped record and nothing else.

A ledger that fails must never take a turn down with it, which is what the bare `except` is for.

In [ ]:
#| export
_HISTORY_LOCKS, _HISTORY_LOCKS_LOCK = {}, threading.RLock()
_SESSION_META_VERSION = 2
LEGACY_GAP = 1800
_SESSION_DEFAULTS = {'title': '', 'title_turns': 0, 'muted': False, 'manual': False}

def _history_lock(agent):
    "One lock per history file, so agents sharing a log serialise their writes to it."
    path = agent.history_path
    if path is None: return agent.__dict__.setdefault('_own_history_lock', threading.RLock())
    key = str(path.resolve())
    with _HISTORY_LOCKS_LOCK: return _HISTORY_LOCKS.setdefault(key, threading.RLock())

@patch(as_prop=True)
def sessions_path(self:Agent):
    return None if self.cfg is None else self.cfg/f'{self.history_name}-sessions.json'

def _session_rows(agent):
    path = agent.sessions_path
    if path is None or not path.exists(): return {}
    try:
        data = json.loads(path.read_text())
        if not isinstance(data, dict): raise ValueError('the document is not an object')
        rows = data.get('sessions', {})
        if not isinstance(rows, dict): raise ValueError('sessions is not an object')
        agent.history_problem = ''
        return rows
    except Exception as e:
        agent.history_problem = f'session metadata is malformed ({agent_err(e)})'
        return None

def _write_session_rows(agent, rows):
    path = agent.sessions_path
    if path is None:return
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f'.{path.name}.{uuid.uuid4().hex}.tmp')
    try:
        tmp.write_text(json.dumps({'version': _SESSION_META_VERSION, 'sessions': rows}, ensure_ascii=False, indent=2) + '\n')
        tmp.replace(path)
    finally:
        if tmp.exists(): tmp.unlink()

_BRANCH_META_VERSION = 1
BRANCH_POLICIES = ('keep', 'discard', 'auto')

@patch(as_prop=True)
def branches_path(self:Agent):
    return None if self.cfg is None else self.cfg/f'{self.history_name}-branches.json'

def _branch_rows(agent):
    "Every recorded branch, or None when the file is there but unreadable."
    path = agent.branches_path
    if path is None or not path.exists(): return {}
    try:
        data = json.loads(path.read_text())
        if not isinstance(data, dict): raise ValueError('the document is not an object')
        rows = data.get('branches', {})
        if not isinstance(rows, dict): raise ValueError('branches is not an object')
        return rows
    except Exception as e:
        agent.history_problem = f'branch metadata is malformed ({agent_err(e)})'
        return None

def _write_branch_rows(agent, rows):
    path = agent.branches_path
    if path is None:return
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f'.{path.name}.{uuid.uuid4().hex}.tmp')
    try:
        tmp.write_text(json.dumps({'version': _BRANCH_META_VERSION, 'branches': rows}, ensure_ascii=False, indent=2) + '\n')
        tmp.replace(path)
    finally:
        if tmp.exists(): tmp.unlink()

@patch
def branch_meta(self:Agent, branch_id=''):
    "One branch's parentage, revision and context manifest. `main` is the conversation as recorded."
    branch_id = str(branch_id or self.current_branch_id)
    with _history_lock(self):
        rows = _branch_rows(self)
        row = {} if rows is None else dict(rows.get(branch_id, {}))
    return {'version': _BRANCH_META_VERSION, 'branch_id': branch_id, 'revision': 0,
            'parent_branch_id': '', 'parent_turn_id': '', 'parent_part_id': '', 'stage': '',
            'created': 0.0, 'shaped': False, 'manifest': {}, 'rewrites': {}} | row

@patch
def branches(self:Agent):
    "Every branch this history knows, newest first, with `main` always present."
    with _history_lock(self): rows = _branch_rows(self) or {}
    out = [self.branch_meta(bid) for bid in rows]
    if 'main' not in rows: out.append(self.branch_meta('main'))
    return sorted(out, key=lambda row: row['created'], reverse=True)

@patch
def save_branch(self:Agent, branch_id, revision=None, **changes):
    """Record one branch. `revision` is the caller's optimistic base: a mismatch means someone
    else moved the branch while a person was deciding, and nothing is written."""
    branch_id = str(branch_id)
    bad = [k for k in changes.get('manifest', {}).values() if k not in BRANCH_POLICIES]
    if bad: raise ValueError(f'unknown context policy {bad[0]!r}')
    with _history_lock(self):
        rows = _branch_rows(self)
        if rows is None: raise ValueError(self.history_problem)
        held = dict(rows.get(branch_id, {}))
        if revision is not None and int(held.get('revision', 0)) != int(revision):
            raise BranchChanged(f'branch {branch_id} moved to revision {held.get("revision", 0)}')
        row = (self.branch_meta(branch_id) | held | dict(changes) |
               {'branch_id': branch_id, 'revision': int(held.get('revision', 0)) + 1})
        row.setdefault('created', time.time())
        if not row['created']: row['created'] = time.time()
        row['shaped'] = (any(p != 'auto' for p in (row.get('manifest') or {}).values())
                         or bool(row.get('rewrites')))
        rows[branch_id] = row
        _write_branch_rows(self, rows)
        return dict(row)

def _index_turn(agent, turn, start, end):
    "Fold one appended turn into the index. The caller holds `_history_lock`, and has just written it."
    sid = turn.get('session') or ''
    if not sid: return
    rows = _session_rows(agent)
    if rows is None: return
    row = dict(rows.get(sid) or {})
    row['turns'] = int(row.get('turns', 0)) + 1
    row['last_at'], row['last_offset'] = turn.get('at', 0), end
    row['model'] = turn.get('model', '')
    if row.get('first_offset') is None:
        row['first_offset'], row['first_at'] = start, turn.get('at', 0)
        row['first_prompt'] = str(turn.get('prompt', '')).replace('\n', ' ')[:72]
    rows[sid] = {'version': _SESSION_META_VERSION, **_SESSION_DEFAULTS} | row
    _write_session_rows(agent, rows)

def _index_stale(agent, rows):
    "Whether the index has to be built again rather than trusted."
    if not rows: return True
    if any(int(r.get('version', 0)) < _SESSION_META_VERSION or 'last_offset' not in r
           for r in rows.values()): return True
    p = agent.history_path
    if p is None or not p.exists(): return False
    # a log smaller than an offset it is supposed to contain was rotated or replaced
    return p.stat().st_size < max((int(r.get('last_offset', 0)) for r in rows.values()), default=0)

def _index_from_log(p):
    "Stream the log a line at a time and describe every conversation in it, with its byte range."
    found, legacy_n, legacy_last, offset = {}, 0, None, 0
    with p.open('rb') as f:
        for raw in f:
            start, offset = offset, offset + len(raw)
            line = raw.decode('utf-8', 'replace').strip()
            if not line: continue
            try: turn = json.loads(line)
            except Exception: continue
            sid, at = turn.get('session') or '', float(turn.get('at') or 0)
            if not sid:
                if legacy_last is None or at - legacy_last > LEGACY_GAP: legacy_n += 1
                sid, legacy_last = f'legacy-{legacy_n}', at
            row = found.setdefault(sid, {'turns': 0, 'first_at': at, 'first_offset': start,
                'first_prompt': str(turn.get('prompt', '')).replace('\n', ' ')[:72]})
            row['turns'] += 1
            row['last_at'], row['last_offset'] = at, offset
            row['model'] = turn.get('model', '')
    return found

def _index_from_history(agent):
    "The same description, for turns held in memory with no file behind them. No offsets to give."
    found, legacy_n, legacy_last = {}, 0, None
    for turn in agent.history:
        sid, at = turn.get('session') or '', float(turn.get('at') or 0)
        if not sid:
            if legacy_last is None or at - legacy_last > LEGACY_GAP: legacy_n += 1
            sid, legacy_last = f'legacy-{legacy_n}', at
        row = found.setdefault(sid, {'turns': 0, 'first_at': at,
            'first_prompt': str(turn.get('prompt', '')).replace('\n', ' ')[:72]})
        row['turns'] += 1
        row['last_at'], row['model'] = at, turn.get('model', '')
    return found

@patch
def rebuild_index(self:Agent, force=False):
    "Indexes conversations in the log with byte ranges, enabling `sessions` and `session_turns` to operate without a full log. If no log exists, in-memory turns constitute the entire conversation."
    with _history_lock(self):
        rows = _session_rows(self)
        if rows is None: return {}          # malformed, reported, and never written over
        p = self.history_path
        logged = p is not None and p.exists()
        if logged and not force and not _index_stale(self, rows): return rows
        found = _index_from_log(p) if logged else _index_from_history(self)
        out = {sid: {'version': _SESSION_META_VERSION, **_SESSION_DEFAULTS}
                    | {k: v for k, v in (rows.get(sid) or {}).items() if k in _SESSION_DEFAULTS}
                    | row
               for sid, row in found.items()}
        for sid, row in rows.items():
            if sid not in out: out[sid] = {**row, 'version': _SESSION_META_VERSION}
        if logged: _write_session_rows(self, out)
        return out

def _session_turns(agent, sid): return agent.session_turns(sid)

@patch
def session_meta(self:Agent, sid=None):
    sid = sid or self.session_id
    with _history_lock(self):
        rows = _session_rows(self)
        row = {} if rows is None else dict(rows.get(sid, {}))
    return {'version': _SESSION_META_VERSION, **_SESSION_DEFAULTS} | row

def _set_session_meta(agent, sid, **changes):
    with _history_lock(agent):
        rows = _session_rows(agent)
        if rows is None: raise ValueError(agent.history_problem)
        row = {'version': _SESSION_META_VERSION, **_SESSION_DEFAULTS} | dict(rows.get(sid, {})) | changes
        rows[sid] = row
        _write_session_rows(agent, rows)
        return dict(row)

@patch
def set_title(self:Agent, text, sid=None):
    sid = sid or self.session_id
    title = ' '.join(str(text).split())[:120]
    return _set_session_meta(self, sid, title=title, title_turns=len(_session_turns(self, sid)),
                             manual=bool(title))

@patch
def set_muted(self:Agent, muted=True, sid=None):
    return _set_session_meta(self, sid or self.session_id, muted=bool(muted))

@patch
def summarize_session(self:Agent, sid=None, force=False):
    sid = sid or self.session_id
    turns, meta = _session_turns(self, sid), self.session_meta(sid)
    if not turns or (meta['manual'] and not force): return meta
    n = len(turns)
    if not force and meta['title_turns'] and n < meta['title_turns'] * 2:return meta
    fallback = meta['title'] or ' '.join(str(turns[0].get('prompt', '')).split())[:72]
    prompt = '\n'.join(str(t.get('prompt', '')) for t in turns[-8:])
    try:
        title = ' '.join(self.oneshot(prompt, 'Write a short conversation title. Output only the title.', 'summary', 32).split())[:120]
        if not title: raise ValueError('the summary model returned no title')
        self.history_problem = ''
    except Exception as e:
        self.history_problem = f'title summary failed ({agent_err(e)})'
        return _set_session_meta(self, sid, title=meta['title'] or fallback)
    return _set_session_meta(self, sid, title=title, title_turns=n, manual=False)

if '_agent_load_history' not in globals():
    _agent_load_history, _agent_remember = Agent._load_history, Agent._remember
    _agent_sessions, _agent_session_turns = Agent.sessions, Agent.session_turns

@patch
def _load_history(self:Agent):
    self.history_problem = ''
    with _history_lock(self): return _agent_load_history(self)

@patch
def refresh_history(self:Agent):
    with _history_lock(self):
        _agent_load_history(self)
        _session_rows(self); _branch_rows(self)
    return self.sessions()

@patch
def session_count(self:Agent):
    "How many turns the whole log holds, from the index rather than from a parse of it."
    return sum(int(r.get('turns', 0)) for r in self.rebuild_index().values())

@patch
def sessions(self:Agent):
    "Lists conversations from the log in reverse order, based on the index. Titles are from user input; silent conversations retain the derived prompt."
    rows = self.rebuild_index()
    if not rows: return _agent_sessions(self)
    return sorted([{'id': sid, 'turns': int(r.get('turns', 0)), 'at': r.get('last_at', 0),
                    'first_at': r.get('first_at', 0), 'model': r.get('model', ''),
                    'title': r.get('title') or r.get('first_prompt', ''),
                    'title_turns': r.get('title_turns', 0), 'muted': bool(r.get('muted', False))}
                   for sid, r in rows.items() if r.get('turns')],
                  key=lambda row: row['at'], reverse=True)

@patch
def session_turns(self:Agent, sid):
    "Reads a conversation's turns from index byte ranges, allowing filtering. Interleaved sessions are handled by reading during the conversation, bounded by its range."
    sid = str(sid)
    row, p = self.rebuild_index().get(sid), self.history_path
    if row is None or 'first_offset' not in row or p is None or not p.exists():
        return _agent_session_turns(self, sid)
    start, end = int(row['first_offset']), int(row.get('last_offset', 0))
    with _history_lock(self):
        with p.open('rb') as f:
            f.seek(start)
            raw = f.read(max(0, end - start))
    mine = (lambda t: not (t.get('session') or '')) if sid.startswith('legacy-') else (lambda t: (t.get('session') or '') == sid)
    out = []
    for line in raw.decode('utf-8', 'replace').splitlines():
        if not line.strip(): continue
        try: turn = json.loads(line)
        except Exception: continue
        if mine(turn): out.append(turn)
    return out

def _ledger_home(agent):
    "The panjika ledger for this agent's project, or None. Resolved once per agent."
    if '_panjika_home' in agent.__dict__: return agent.__dict__['_panjika_home']
    home = None
    try:
        from panjika.core import Home
        roots = getattr(getattr(agent, 'host', None), 'roots', None) or []
        found = Home(start=str(roots[0]) if roots else Path.cwd())
        home = found if found.exists else None
    except Exception: home = None
    agent.__dict__['_panjika_home'] = home
    return home

def _to_ledger(agent, turn):
    "Hand one turn to panjika, when the project keeps a ledger and panjika is installed."
    home = _ledger_home(agent)
    if home is None or not turn.get('session'): return
    try:
        from panjika.harness import ingest
        ingest({**turn, 'cwd': str(home.root)}, 'ramabana', home)
    except Exception: pass

@patch
def _remember(self:Agent, prompt, text, error='', state='complete'):
    with _history_lock(self):
        _agent_remember(self, prompt, text, error, state)
        if (span := self.__dict__.pop('_last_span', None)) and self.history: _index_turn(self, self.history[-1], *span)
    if self.history: _to_ledger(self, self.history[-1])

A turn reaches the ledger only where one exists, so an agent run in a project that never asked for panjika leaves nothing behind. Where one does exist the turn arrives whole: the ask, the reply, the tokens, and every file a tool moved, which is what `landed` then answers about.

In [ ]:
import subprocess, tempfile
from panjika.core import Home
from panjika.git import landed
from panjika.read import Ledger
from ramabana.testing import MemHost, fake_agent

def _git_(root, *a): subprocess.run(['git', *a], cwd=root, capture_output=True, check=True)

d = Path(tempfile.mkdtemp())/'proj'; d.mkdir(parents=True)
_git_(d, 'init', '-q', '-b', 'main')
_git_(d, 'config', 'user.email', 'a@b.c'); _git_(d, 'config', 'user.name', 'Sam')
(d/'charges.py').write_text('def total(a):\n    return sum(a)\n')
_git_(d, 'add', '-A'); _git_(d, 'commit', '-qm', 'billing')

a, _ = fake_agent(MemHost({}, root=str(d)), replies=['nothing to do'])
a.session_id = 'rb-1'
test_eq(a.ask('look around'), 'nothing to do')
test_eq(_ledger_home(a), None)
assert not (d/'.panjika').exists()

Home(d/'.panjika').init()
b, _ = fake_agent(MemHost({}, root=str(d)), replies=['rounded the total'])
b.session_id = 'rb-1'
b.ask('round the total to 2dp')
row = Ledger(d/'.panjika').session('rb-1')
test_eq((row.harness, row.status, row.prompt), ('ramabana', 'complete', 'round the total to 2dp'))
test_eq([n.text for n in row.notes], ['rounded the total'])

(d/'charges.py').write_text('def total(a):\n    return round(sum(a), 2)\n')
_to_ledger(b, {'at': time.time(), 'session': 'rb-1', 'state': 'complete', 'model': 'sonnet',
               'prompt': 'round the total to 2dp', 'reply': 'rounded it', 'error': '',
               'usage': {'input': 1840, 'output': 96},
               'activity': [{'tool': 'edit_file', 'args': {'path': str(d/'charges.py')},
                             'ok': True, 'secs': 0.4, 'summary': 'edited charges.py'}]})
v = landed('rb-1', home=d/'.panjika', start=d)[0]
test_eq((v.state, v.path, v.branch), ('pending', 'charges.py', 'main'))
test_eq(Ledger(d/'.panjika').session('rb-1').tokens_in, 1840)

A log shared by a frontend and the CLI reaches tens of megabytes, so `_load_history` reads its last `HISTORY_TAIL` bytes rather than all of it. Whichever of the two bounds bites first wins, so a log inside the window behaves exactly as it did before there was one.

The tail is what the model's context is rebuilt from. It is not what a picker needs: the conversations worth offering are mostly older than the window. `rebuild_index` streams the log once and records, per session, how many turns it holds and which bytes it occupies. `session_turns` seeks to that range: a range with a filter rather than a list of offsets, because two live conversations interleave in one log and a session's lines are not contiguous.

In [ ]:
import json as _json, tempfile as _tf
_cfg = Path(_tf.mkdtemp())
_pad = 'x'*400
_rows = ([{'at':1,'session':'old','prompt':f'ancient {_pad}','reply':'r','model':'gpt-mini'}] +
         [{'at':10+i,'session':'new','prompt':f'{i} {_pad}','reply':'r','model':'gpt-mini'} for i in range(30)])
(_cfg/'tail-history.jsonl').write_text('\n'.join(_json.dumps(r) for r in _rows)+'\n')

_hold, HISTORY_TAIL = HISTORY_TAIL, 2000
_a = Agent(NullHost(['.']), cfg=_cfg, history_name='tail', extensions=False)
test_eq({t['session'] for t in _a.history}, {'new'})       # `old` is outside the window
test_eq([s['id'] for s in _a.sessions()], ['new', 'old'])  # and still in the picker
test_eq(_a.sessions()[1]['turns'], 1)
test_eq(_a.session_turns('old')[0]['prompt'][:7], 'ancient')
HISTORY_TAIL = _hold

In [ ]:
#| export
@patch
def add_chat_callback(self:Agent, name):
    "Attaches a named Rishi callback to the turn chat and its replacements. The registry contains the callbacks to attach, not the catalogue; seeding with all callbacks causes `_be` to attach all of them to each chat."
    from .runtime import CHAT_CALLBACKS
    held = getattr(self, '_chat_callbacks', None)
    if held is None: held = self._chat_callbacks = {}
    if name in held: return name
    if name not in CHAT_CALLBACKS: raise KeyError(f'unknown callback {name!r}; known: {", ".join(sorted(CHAT_CALLBACKS))}')
    held[name] = CHAT_CALLBACKS[name]
    self.backend.add_cb(held[name])
    return name

@patch
def chat_callback(self:Agent, name):
    "Attach one built-in callback to the current turn backend."
    return self.add_chat_callback(name)

In [ ]:
#| export
@patch
def _be(self:Agent, job='turn'):
    "The backend for `job`, applying registered turn callbacks when it is first built."
    spec = self.routing.spec(job)
    key = (spec.backend, spec.model_id)
    if key not in self._backends:
        is_turn = key == (lambda s: (s.backend, s.model_id))(self.routing.spec('turn'))
        kw = {'multimodal': self.local_multimodal} if spec.runtime == 'litert' else {}
        if is_turn:
            kw.update(sp=self.system_prompt(), tools=self.tools, tool_max_len=self.tool_max_len,
                      approve=(self.approvals.gate if self.approvals is not None else None))
        backend = make_backend(spec, **kw)
        if is_turn:
            for cb in getattr(self, '_chat_callbacks', {}).values(): backend.add_cb(cb)
        self._backends[key] = backend
    return self._backends[key]

`status` returns routing, backend readiness, and current problems for status-bar rendering.

In [ ]:
# an agent built after the budget cells above, so it carries `tool_budget` and `step_budget`
bar = demo_agent()
{k: v for k, v in bar.status().items() if k in ('ready', 'busy', 'model', 'tool_budget')}

In [ ]:
st = bar.status()
assert {'ready', 'busy', 'model', 'tool_budget', 'step_budget'} <= set(st), sorted(st)
test_eq(st['busy'], False)
test_eq(st['ready'], False)                                # nothing asked yet
assert 'claude-sonnet-4-5' in str(st['model'])

Slash commands live here rather than in a frontend, because the CLI and the editor both need them
and neither should own them.

In [ ]:
bar.command('/tools').splitlines()[:3]

In [ ]:
assert 'search_code' in bar.command('/tools')
assert bar.command('/model')
test_eq(bar.command('/nope'), None)                        # an unknown command is not this agent's
test_eq(bar.command('not a command'), None)

The provider messages for a reshaped conversation: the stored turns, minus what a person discarded,
with their own words in place of any prose they rewrote. A call and the result it produced move
together, and neither can be rewritten, because editing one would claim work that never ran.

In [ ]:
shaper.compile_conversation()['messages']

In [ ]:
whole = shaper.compile_conversation()['messages']
test_eq([m['role'] for m in whole], ['user', 'assistant'])
# a manifest says keep, discard or auto per part -- not a boolean, so a typo is caught
pid = shaper.conversation_parts()[-1]['part_id']
test_fail(lambda: shaper.compile_conversation(manifest={pid: False}), contains='unknown context policy')
fewer = shaper.compile_conversation(manifest={pid: 'discard'})['messages']
test_eq([m['role'] for m in fewer], ['user'])
mine = shaper.compile_conversation(rewrites={pid: 'Cyan, actually.'})['messages']
assert 'Cyan, actually.' in str(mine[-1])

Makes a reshaped conversation the active branch, without touching what was recorded. Reshaping is
not editing history: the turn log still says what happened.

In [ ]:
pid = shaper.conversation_parts()[-1]['part_id']
shaper.reshape(rewrites={pid: 'Cyan, actually.'})

In [ ]:
logged = len(shaper.history)
pid = shaper.conversation_parts()[-1]['part_id']
br = shaper.reshape(rewrites={pid: 'Cyan, actually.'})
assert br['branch_id'] and br['branch_id'] != 'main'
test_eq(shaper.current_branch_id, br['branch_id'])
test_eq(len(shaper.history), logged)                       # the record is untouched

A branch is its parent point plus its manifest, so switching means recompiling rather than copying.
`main` is always there to come back to.

In [ ]:
shaper.switch_branch('main')
shaper.current_branch_id, [b['branch_id'] for b in shaper.branches()]

In [ ]:
shaper.switch_branch('main')
test_eq(shaper.current_branch_id, 'main')
ids = [b['branch_id'] for b in shaper.branches()]
assert 'main' in ids, ids
test_eq(shaper.switch_branch('main')['branch_id'], 'main')  # switching to the current one is a no-op

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()